# Tahap 2b — Uji plafon routing & gate post-hoc (detektor beku), 3 seed dalam satu notebook

**Latar.** Tahap 2: H2 gagal di 3 seed. Routing *oracle-loss* (target training gate, loss_A < loss_B) pun tidak lebih baik dari acak dan lebih buruk pada mAP_small. Jadi masalahnya ada di target, tidak hanya di gate.

**Pertanyaan notebook ini**
1. **Plafon**: bila target diukur dari *hasil deteksi* (bukan loss), adakah routing per gambar yang jauh lebih baik dari acak pada aktivasi yang sama? Oracle memakai GT test, jadi ini **batas atas** (termasuk optimisme seleksi), bukan angka deployment.
2. **Bisa dipelajari?** Gate baru di atas detektor **beku**. Inputnya fitur pooled (GAP + GMP) dari keluaran Stage 1 (p3_neck, p2_backbone, p4_neck, p5_sppf) + logit gate lama z, yang sudah ada di engine Stage 1, jadi bisa di-deploy.
   - Dilatih di **train**, dipilih di **val** dengan aturan tetap (argmax rata-rata gain mAP_small vs acak pada ρ ∈ {0.2, 0.4, 0.6, 0.8}), lalu dinilai **sekali** di **test**.
   - Kandidat: 3 target × 8 model (Ridge, LogReg, HGB).

**Aturan yang dikunci sebelum melihat test**
- Titik uji utama **ρ = 0.5**, ditambah r\* Tahap 2 per seed sebagai pembanding.
- Uji: bootstrap berpasangan per gambar (B = 1000) terhadap rata-rata 5 undian acak pada ρ yang sama.
- Metrik utama mAP_small COCO (sama dengan H2).

**Target per gambar** (dari evalImgs COCO, IoU 0.5, Δ = A − B)
- `soft_small`: jumlah skor deteksi TP pada GT kecil.
- `soft_all`: sama, untuk semua objek.
- `net25_all`: TP − FP semua objek pada conf ≥ 0.25 (ambang 0.25 adalah asumsi; dua target lain bebas ambang).

**Format input** = letterbox persegi 640 (format deployment), sama dengan Tahap 2. Deteksi: conf 0.001, IoU 0.7, max_det 300, koordinat dikembalikan ke resolusi asli per sumbu.

**Prasyarat**: GPU T4, Internet On, dataset `kitti-2class`, ketiga arsip Tahap 1 (sama dengan Tahap 2). Perkiraan ±20–25 menit per seed. Hasil per seed disimpan segera, jadi bila sesi putus, menjalankan ulang melewati seed yang sudah selesai.

In [ ]:
# [1] KONFIGURASI
import os, sys, json, time, shutil, subprocess, hashlib, platform, datetime
from pathlib import Path

CFG = dict(
    repo_url="https://github.com/maugi1547/thesis-Difficulty-Aware-Router.git",
    repo_dir="/kaggle/working/thesis-Difficulty-Aware-Router",
    code_commit="5416543",
    data_yaml="/kaggle/input/datasets/agungmaugi/kitti-2class/kitti_2class.yaml",
    model_yaml="ultralytics/cfg/models/v8/yolov8n-p2-router.yaml",
    archives={42: "/kaggle/input/datasets/agungmaugi/archive-m0-lam002-seed42",
              7: "/kaggle/input/datasets/agungmaugi/archive-m0-lam002-seed7",
              2026: "/kaggle/input/datasets/arungmaugi/archive-m0-lam002-seed2026"},
    r_star_tahap2={42: 0.6, 7: 0.8, 2026: 0.6},      # dari calib.json Tahap 2 (pembanding)
    ckpt_name="last.pt",
    out_dir="/kaggle/working/tahap2b",
    imgsz=640, conf=0.001, iou=0.7, max_det=300,
    rhos=[0.2, 0.4, 0.6, 0.8],      # grid seleksi di val & kurva
    rho_test=0.5,                   # titik uji utama (dikunci)
    select_metric="mAP_small",
    n_train=None,                   # None = seluruh train; isi angka untuk uji cepat
    n_random_curve=20, n_random_boot=5, n_boot=1000,
    expected_counts={"val": 1297, "test": 1000},
)
CFG.update(globals().get("CFG_OVERRIDE", {}))
assert CFG["repo_dir"] == "/kaggle/working/thesis-Difficulty-Aware-Router" or "CFG_OVERRIDE" in globals()
REPO = Path(CFG["repo_dir"]); OUT = Path(CFG["out_dir"]); OUT.mkdir(parents=True, exist_ok=True)
PROV = {"started_utc": datetime.datetime.utcnow().isoformat(timespec="seconds"), "cfg": CFG}
T_START = time.perf_counter()

def sh(cmd, cwd=None, secret=None):
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    if secret:
        out = out.replace(secret, "***")
    if r.returncode != 0:
        raise RuntimeError(f"{' '.join(c if c != secret else '***' for c in cmd)}\n{out}")
    return out

print(json.dumps(CFG, indent=1, default=str))

In [ ]:
# [2] CLONE dari GitHub + checkout commit terkunci (clone baru setiap sesi -> tidak ada sisa state lama)
token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass                                              # repo publik: token tidak perlu
url = CFG["repo_url"]
auth_url = url.replace("https://", f"https://x-access-token:{token}@") if token else url

if REPO.exists():
    shutil.rmtree(REPO)
sh(["git", "clone", "--quiet", auth_url, str(REPO)], secret=token)
sh(["git", "remote", "set-url", "origin", url], cwd=REPO)          # token tidak tersimpan di .git/config
try:
    sh(["git", "checkout", "--quiet", CFG["code_commit"]], cwd=REPO)
except RuntimeError as e:
    raise RuntimeError(f"Commit {CFG['code_commit']} tidak ada di GitHub. Sudah di-push? "
                       f"HEAD remote = {sh(['git', 'log', '-1', '--format=%h %s', 'origin/HEAD'], cwd=REPO)}") from e
PROV["git_commit"] = sh(["git", "rev-parse", "HEAD"], cwd=REPO)
PROV["git_commit_msg"] = sh(["git", "log", "-1", "--format=%s (%cd)"], cwd=REPO)
PROV["git_branches_containing"] = sh(["git", "branch", "-r", "--contains", "HEAD"], cwd=REPO).split()
print("✅ commit:", PROV["git_commit"][:10], "|", PROV["git_commit_msg"], "| branch:", PROV["git_branches_containing"],
      "| token:", "ya" if token else "tidak")

In [ ]:
# [3] INSTALL editable dari repo (ultralytics pip bawaan dihapus agar tidak tertukar)
print(sh([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "ultralytics"]) or "uninstall: ok")
print(sh([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO)])[-500:] or "install: ok")

In [ ]:
# [3b] Paket evaluasi
print(sh([sys.executable, "-m", "pip", "install", "-q", "pycocotools", "scikit-learn"])[-500:] or "install: ok")

In [ ]:
%%writefile /kaggle/working/thesis-Difficulty-Aware-Router/router_train_utils.py
# [4] versi sama dengan Tahap 1 (hapus sel ini setelah file di-commit ke repo)
"""
router_train_utils.py — utilitas murni untuk training & sanity check Router-P2 Dual-Branch.

Dipakai bersama oleh script_training.py (run utama) dan notebook smoke test, supaya yang
diuji di Tahap 0 adalah kode yang sama dengan yang dipakai run 100 epoch.
Isi: scheduler berparameter, penerap ROUTER_CFG, logger CSV router (per-run, 11 kolom,
label epoch diperbaiki), logger loss + penjaga NaN, timer epoch, pemeriksa CSV/loss,
estimasi anggaran, uji kalibrasi tersimpan, uji konsistensi gate.
"""
import os, sys, csv, json, time, shutil
from pathlib import Path
from copy import deepcopy

import numpy as np
import pandas as pd
import torch

CSV_COLS = ["Epoch", "Batch", "Lambda", "P2_Prob_Real", "Rel", "Diff_W", "Final_L3",
            "GV_Target", "GV_Loss", "GV_Weight", "GV_Offset"]
LOSS_NAMES = ["box_A", "cls_A", "dfl_A", "router", "proxy", "box_B", "cls_B", "dfl_B"]


def find_router(model):
    """Kembalikan modul router pertama (nama kelas mengandung 'DifficultyAwareRouter')."""
    for m in model.modules():
        if "DifficultyAwareRouter" in m.__class__.__name__:
            return m
    raise RuntimeError("Router tidak ditemukan di model.")


def _ramp(epoch, start, length, final):
    """0 sampai epoch == start, lalu naik linear ke `final` selama `length` epoch, lalu tetap."""
    if epoch <= start:
        return 0.0
    return final * min((epoch - start) / max(length, 1), 1.0)


def make_curriculum_scheduler(router_warmup_end, gv_start, gv_ramp, final_gv,
                              lam_start, lam_ramp, final_lambda, history=None):
    """
    Versi berparameter dari ideal_curriculum_scheduler (script_training.py).

    Konfigurasi run utama yang IDENTIK dengan kode lama:
        make_curriculum_scheduler(router_warmup_end=10, gv_start=10, gv_ramp=20, final_gv=0.5,
                                  lam_start=50, lam_ramp=30, final_lambda=<nilai Anda>)
    Perbedaan perilaku dgn versi lama: _is_warmup di-set dua arah (lama hanya mematikan).
    Untuk run baru hasilnya sama; untuk resume, versi ini lebih benar.

    history: dict opsional, diisi {epoch: {"lambda", "gv", "warmup"}} untuk diverifikasi.
    """
    def scheduler(trainer):
        epoch = trainer.epoch + 1
        lam = _ramp(epoch, lam_start, lam_ramp, final_lambda)
        gv = _ramp(epoch, gv_start, gv_ramp, final_gv)
        warm = epoch <= router_warmup_end
        targets = [trainer.model]
        if getattr(trainer, "ema", None):
            targets.append(trainer.ema.ema)
        for mdl in targets:
            mdl.router_penalty_lambda = lam
            mdl.gate_value_loss_weight = gv
            mdl.total_batches = len(trainer.train_loader)
            find_router(mdl)._is_warmup = warm
        if history is not None:
            history[epoch] = {"lambda": lam, "gv": gv, "warmup": warm}
        print(f"📈 [SCHEDULER] epoch {epoch} | warmup={warm} | lambda={lam:.4f} | gate_value_w={gv:.4f}")
    return scheduler


def make_router_cfg_applier(router_cfg, dump_path=None):
    """
    Callback on_pretrain_routine_end: tulis hyperparameter router secara EKSPLISIT ke model
    (bukan mengandalkan default getattr yang tersebar di loss.py & model_yolov8_router.py).
    Harus jalan sebelum batch pertama, karena DualBranchDetectionLoss membaca
    branch_b_loss_weight & gate_offset_momentum saat criterion dibuat (lazy, batch pertama).
    """
    def apply(trainer):
        for mdl in [trainer.model] + ([trainer.ema.ema] if getattr(trainer, "ema", None) else []):
            for k, v in router_cfg.items():
                setattr(mdl, k, v)
        if dump_path is not None:
            Path(dump_path).write_text(json.dumps(router_cfg, indent=1))
        print("🔧 [ROUTER CFG]", router_cfg)
    return apply


def make_router_csv_logger(csv_path, allow_existing=False):
    """
    Pengganti on_train_epoch_end di script_training.py:
      - path per-run (bukan /kaggle/working/... global yang di-append lintas run),
      - menolak file yang sudah ada (cegah data run lama tercampur),
      - memverifikasi tiap baris berisi 11 kolom,
      - kolom Epoch diganti epoch ASLI (label di loss.py = debug_counter//total_batches + 4,
        salah setiap kali warmup tidak berakhir di epoch 4). Tetap 11 kolom.
    """
    csv_path = Path(csv_path)
    if csv_path.exists() and not allow_existing:      # allow_existing=True hanya untuk resume run yang sama
        raise FileExistsError(f"{csv_path} sudah ada — hapus/ganti nama dulu.")

    def on_train_epoch_end(trainer):
        buf = getattr(trainer.model, "router_buffer", None)
        if not buf:
            return
        epoch = trainer.epoch + 1
        new = not csv_path.exists()
        with open(csv_path, "a", newline="") as f:
            w = csv.writer(f)
            if new:
                w.writerow(CSV_COLS)
            for row in buf:
                if len(row) != len(CSV_COLS):
                    raise ValueError(f"router_buffer berisi {len(row)} kolom, header {len(CSV_COLS)}: {row}")
                w.writerow([epoch] + list(row[1:]))
        buf.clear()
    return on_train_epoch_end


def make_loss_items_logger(store, csv_path=None):
    """
    on_train_batch_end: simpan 8 komponen loss per batch + GAGAL CEPAT bila NaN/Inf.
    (Ultralytics 8.3.240 punya NaN recovery yang diam-diam memuat ulang checkpoint.)
    Bila csv_path diberikan, `store` ditulis ke file & dikosongkan lewat .flush(trainer)
    (dipasang di on_train_epoch_end) supaya run 100 epoch tidak menumpuk di RAM.
    """
    state = {"header": csv_path is not None and not Path(csv_path).exists()}

    def on_train_batch_end(trainer):
        li = trainer.loss_items.detach().float().cpu()
        epoch = trainer.epoch + 1
        store.append([epoch] + li.tolist())
        if not torch.isfinite(li).all() or not torch.isfinite(trainer.loss.detach()).all():
            raise FloatingPointError(f"Loss non-finite di epoch {epoch}, batch #{len(store)}: "
                                     f"{dict(zip(LOSS_NAMES, li.tolist()))}")

    def flush(trainer):
        if csv_path is None or not store:
            return
        pd.DataFrame(store, columns=["epoch"] + LOSS_NAMES).to_csv(
            csv_path, mode="a", header=state["header"], index=False)
        state["header"] = False
        store.clear()

    on_train_batch_end.flush = flush
    return on_train_batch_end


class EpochTimer:
    """
    Ukur waktu per epoch, dipisah: train (loop batch), val_total (validasi + save + callback),
    dan valB (validasi Branch B, diukur lewat wrapper `timed`).
    cuda.synchronize() supaya kernel async tidak 'bocor' ke fase berikutnya.
    """
    def __init__(self):
        self.rows, self._t, self._valB = [], {}, 0.0

    @staticmethod
    def now():
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        return time.perf_counter()

    def on_train_epoch_start(self, trainer):
        self._t["t0"] = self.now()
        self._valB = 0.0

    def on_train_epoch_end(self, trainer):
        self._t["t1"] = self.now()

    def on_fit_epoch_end(self, trainer):
        # final_eval() Ultralytics memanggil on_fit_epoch_end SEKALI LAGI setelah loop
        # (validasi best.pt) -> abaikan panggilan yang tidak didahului on_train_epoch_end.
        if self._t.get("t1") is None:
            return
        t2 = self.now()
        self.rows.append(dict(epoch=trainer.epoch + 1,
                              n_batches=len(trainer.train_loader),
                              train_s=self._t["t1"] - self._t["t0"],
                              val_total_s=t2 - self._t["t1"],
                              valB_s=self._valB))
        self._t["t1"] = None

    def timed(self, cb):
        def wrapper(validator):
            t0 = self.now()
            cb(validator)
            self._valB += self.now() - t0
        return wrapper


# ---------------------------- pemeriksa ----------------------------
class Report:
    def __init__(self):
        self.items = []

    def add(self, name, status, detail=""):
        icon = {"PASS": "✅", "WARN": "⚠️ ", "FAIL": "❌"}[status]
        print(f"{icon} [{status}] {name}" + (f" — {detail}" if detail else ""))
        self.items.append({"check": name, "status": status, "detail": str(detail)})

    def check(self, name, ok, detail="", warn_only=False):
        self.add(name, "PASS" if ok else ("WARN" if warn_only else "FAIL"), detail)

    @property
    def n_fail(self):
        return sum(i["status"] == "FAIL" for i in self.items)


def check_router_csv(df, n_batches, epochs, router_warmup_end, sched_history,
                     offset_clip=1.0, report=None):
    """
    Semua pemeriksaan pada router_loss_components.csv. `df` dibaca dgn pd.read_csv.

    Artefak yang DIKETAHUI dan diperhitungkan:
      - Sejak AUDIT-FIX #6 kolom GV_* berasal dari step yang SAMA (tidak lagi tertinggal 1 step),
        jadi pemeriksaan di sini ketat untuk semua baris. Dengan kode lama, baris pertama
        pasca-warmup akan gagal di cek GV_Weight (nilai warmup = 0).
      - GV_Offset adalah offset yang DIPAKAI step ini (nilai EMA sebelum di-update). Baris
        pertama pasca-warmup = 0 (EMA belum terinisialisasi), ini normal.
      - GV_Offset dicetak 4 desimal -> perubahan EMA < 5e-5 tampak 'tidak berubah'.
      - Mode 'small': batch tanpa gambar valid (tidak ada objek kecil di KEDUA branch)
        -> GV_Loss = GV_Target = 0 dan offset tidak di-update.
    """
    r = report or Report()
    r.check("CSV: header 11 kolom dgn urutan benar", list(df.columns) == CSV_COLS, list(df.columns))
    if list(df.columns) != CSV_COLS:
        return r

    exp_rows = (epochs - router_warmup_end) * n_batches
    r.check("CSV: jumlah baris = epoch pasca-warmup × batch/epoch", len(df) == exp_rows,
            f"{len(df)} vs {exp_rows}")
    r.check("CSV: tidak ada baris pada epoch warmup", int((df.Epoch <= router_warmup_end).sum()) == 0)

    num = df[CSV_COLS[2:]].astype(float)
    r.check("CSV: semua nilai finite (tidak NaN/Inf)", bool(np.isfinite(num.values).all()))

    # --- jadwal tercatat sesuai scheduler ---
    for e, g in df.groupby("Epoch"):
        h = sched_history.get(int(e))
        if h is None:
            r.add(f"E{e}: riwayat scheduler", "FAIL", "epoch tidak ada di riwayat scheduler")
            continue
        r.check(f"E{e}: kolom Lambda = jadwal ({h['lambda']:.4f})",
                np.allclose(g.Lambda.astype(float), h["lambda"], atol=1e-4))
        r.check(f"E{e}: kolom GV_Weight = jadwal ({h['gv']:.4f})",
                np.allclose(g.GV_Weight.astype(float), h["gv"], atol=1e-4))
        if h["lambda"] == 0:
            r.check(f"E{e}: Final_L3 = 0 saat lambda = 0", bool((g.Final_L3.astype(float) == 0).all()))
        else:
            r.check(f"E{e}: Final_L3 > 0 saat lambda > 0", bool((g.Final_L3.astype(float) > 0).all()),
                    f"min={g.Final_L3.min():.6f}")

    # --- rentang yang dijamin oleh kode ---
    r.check("P2_Prob_Real di [0, 1]", bool(df.P2_Prob_Real.between(0, 1).all()))
    r.check("Diff_W di [0.1, 2.0] (clamp di compute_router_loss)", bool(df.Diff_W.between(0.1 - 1e-6, 2.0 + 1e-6).all()),
            f"[{df.Diff_W.min():.4f}, {df.Diff_W.max():.4f}]")
    r.check("Rel ≥ 0", bool((df.Rel >= 0).all()))
    r.check("|GV_Loss| ≤ clip advantage (mean(p·adv), |adv| ≤ clip)",
            bool((df.GV_Loss.abs() <= offset_clip + 1e-6).all()), f"max |GV_Loss| = {df.GV_Loss.abs().max():.4f}")

    # --- kolom GV berubah tiap batch (baris pertama: offset EMA belum ada) ---
    g = df.iloc[1:].reset_index(drop=True)
    empty = ((g.GV_Loss == 0) & (g.GV_Target == 0)).mean()
    r.check("Batch tanpa sampel valid (GV_Loss = GV_Target = 0) < 5%", empty < 0.05, f"{empty:.1%}", warn_only=True)
    for col, thr in [("GV_Target", 0.95), ("GV_Loss", 0.95), ("GV_Offset", 0.80)]:
        frac = float((g[col].diff().iloc[1:] != 0).mean()) if len(g) > 1 else 0.0
        r.check(f"{col} berubah antar-batch (≥ {thr:.0%} batch)", frac >= thr, f"{frac:.1%}")

    # --- GV_Offset tidak menempel di ±clip ---
    at_clip = float((df.GV_Offset.abs() >= offset_clip - 1e-3).mean())
    r.check(f"GV_Offset tidak menempel di ±{offset_clip}", at_clip == 0, f"{at_clip:.1%} baris di batas")
    r.check("GV_Offset jauh dari batas (|offset| < 0.5·clip)", bool(df.GV_Offset.abs().max() < 0.5 * offset_clip),
            f"rentang [{df.GV_Offset.min():.4f}, {df.GV_Offset.max():.4f}]", warn_only=True)
    tail = df.GV_Offset.iloc[-max(len(df) // 4, 2):]
    r.check("GV_Offset tidak monoton naik/turun terus di 25% akhir (indikasi drift)",
            not (tail.diff().dropna() > 0).all() and not (tail.diff().dropna() < 0).all(),
            f"awal {tail.iloc[0]:.4f} → akhir {tail.iloc[-1]:.4f}", warn_only=True)

    # --- aktivasi tidak kolaps (indikator awal saja, 1 epoch tidak konklusif) ---
    last_e = df.Epoch.max()
    p_last = float(df[df.Epoch == last_e].P2_Prob_Real.mean())
    r.check(f"E{last_e}: rata-rata P2_Prob_Real di [0.05, 0.95] (bukan kolaps)", 0.05 <= p_last <= 0.95,
            f"{p_last:.3f}", warn_only=True)
    return r


def check_loss_items(li_df, router_warmup_end, sched_history, report=None):
    """Pemeriksaan 8 komponen loss per batch (loss_items.csv)."""
    r = report or Report()
    vals = li_df[LOSS_NAMES].astype(float)
    r.check("loss_items: 8 komponen finite di semua batch", bool(np.isfinite(vals.values).all()))
    for e, g in li_df.groupby("epoch"):
        lam = sched_history.get(int(e), {}).get("lambda", 0.0)
        warm = int(e) <= router_warmup_end
        rmean = float(g.router.mean())
        if warm or lam == 0:
            r.check(f"E{e}: komponen 'router' = 0 (warmup / lambda 0)", bool((g.router == 0).all()), f"mean={rmean:.6f}")
        else:
            r.check(f"E{e}: komponen 'router' > 0 (lambda > 0)", rmean > 0, f"mean={rmean:.6f}")
    # informatif (bukan kriteria lulus): 3 epoch terlalu pendek untuk menilai konvergensi
    print(li_df.groupby("epoch")[LOSS_NAMES].mean().round(4).to_string())
    return r


def budget_table(timing_df, plan, branch_b_interval=10, overhead=0.10, session_limit_h=12.0):
    """
    Estimasi jam GPU per run dari waktu epoch 'steady' (epoch ≥ 2; epoch 1 memuat cache,
    autotune cuDNN, check_amp, dsb).

    jam = (E·(t_train + t_valA) + floor(E/interval_B)·t_valB) · (1 + overhead) / 3600
    t_valA = val_total − valB (validasi Branch A + simpan checkpoint + callback lain).
    """
    st = timing_df[timing_df.epoch >= 2]
    if st.empty:
        st = timing_df
    t_train = st.train_s.mean()
    t_valB = st.valB_s.mean()
    t_valA = (st.val_total_s - st.valB_s).mean()
    rows = []
    for name, (E, n_runs) in plan.items():
        h = (E * (t_train + t_valA) + (E // branch_b_interval) * t_valB) * (1 + overhead) / 3600
        rows.append(dict(run=name, epochs=E, n_runs=n_runs, jam_per_run=round(h, 2),
                         jam_total=round(h * n_runs, 2),
                         perlu_resume=h > session_limit_h - 0.5))
    summary = dict(t_train_s=round(t_train, 1), t_valA_s=round(t_valA, 1), t_valB_s=round(t_valB, 1),
                   epoch1_train_s=round(float(timing_df.iloc[0].train_s), 1))
    return pd.DataFrame(rows), summary


CALIB_T_TEST = 1.2345678901234567   # nilai "tidak bulat" -> pembulatan fp16/fp32 pasti ketahuan
CALIB_B_TEST = -0.3456789012345678


def _calib(router):
    return float(getattr(router, "calib_T", 1.0)), float(getattr(router, "calib_b", 0.0))


def test_calibration_persistence(model, build_fresh_model, tmp_dir, report=None):
    """
    Uji 0.4. Set calib_T/calib_b, simpan, muat ulang lewat 3 jalur yang benar-benar dipakai:
      J1 pickle checkpoint Ultralytics : torch.save({'ema': deepcopy(model).half()}) -> torch.load
         (= trainer.save_model + attempt_load_one_weight)
      J2 state_dict -> model baru       : load_state_dict(strict=False)
         (= load_dualbranch() di notebook uji-akurasi-latensi)
      J3 Ultralytics model.load()       : intersect_dicts + load_state_dict
         (= DualBranchTrainer.get_model(weights=...), mis. fine-tune)

    Sejak AUDIT-FIX #9 kalibrasi disimpan sebagai buffer int64 dalam mikro-unit, jadi nilai
    yang tersimpan = round(v·1e6)/1e6. "Sama persis" di sini berarti: nilai yang DIBACA
    sebelum simpan == nilai setelah muat ulang (==, bukan allclose), dan galat kuantisasi
    set→baca ≤ 5e-7.
    """
    r = report or Report()
    tmp_dir = Path(tmp_dir); tmp_dir.mkdir(parents=True, exist_ok=True)
    rt = find_router(model)
    rt.calib_T, rt.calib_b = CALIB_T_TEST, CALIB_B_TEST
    ref = (rt.calib_T, rt.calib_b)
    r.check("Kalibrasi: galat kuantisasi set→baca ≤ 5e-7",
            abs(ref[0] - CALIB_T_TEST) <= 5e-7 and abs(ref[1] - CALIB_B_TEST) <= 5e-7,
            f"set=({CALIB_T_TEST}, {CALIB_B_TEST}) baca={ref}")
    keys = [k for k in model.state_dict() if "calib" in k]
    r.check("Kalibrasi ikut state_dict (buffer)", len(keys) == 2, keys)

    # J1
    p = tmp_dir / "calib_pickle.pt"
    torch.save({"ema": deepcopy(model).half()}, p)
    m1 = torch.load(p, map_location="cpu", weights_only=False)["ema"].float()
    rt1 = find_router(m1)
    r.check("Kalibrasi J1 (pickle checkpoint Ultralytics, lewat .half()) identik", _calib(rt1) == ref, f"{_calib(rt1)}")
    r.check("Buffer kalibrasi tetap int64 setelah .half()/.float()",
            all(b.dtype == torch.int64 for n, b in rt1.named_buffers() if "calib" in n))

    # J2
    sd = {k: (v.float() if v.is_floating_point() else v) for k, v in model.state_dict().items()}
    torch.save(sd, tmp_dir / "calib_state_dict.pt")
    sd2 = torch.load(tmp_dir / "calib_state_dict.pt", map_location="cpu")
    m2 = build_fresh_model()
    missing, unexpected = m2.load_state_dict(sd2, strict=False)
    r.check("Kalibrasi J2 (state_dict -> model baru) identik", _calib(find_router(m2)) == ref,
            f"{_calib(find_router(m2))} | missing={len(missing)} unexpected={len(unexpected)}")

    # J3
    m3 = build_fresh_model()
    m3.load(m1, verbose=False)
    r.check("Kalibrasi J3 (Ultralytics model.load) identik", _calib(find_router(m3)) == ref, f"{_calib(find_router(m3))}",
            warn_only=True)   # jalur fine-tune: ikut mewarisi kalibrasi -> sadari saat fine-tune
    return r, (m1, m2, m3)


@torch.no_grad()
def gate_decisions_both_paths(model, img):
    """
    Satu forward eval penuh (model(img) -> _predict_once -> router.forward eval), tangkap input
    router via forward-pre-hook, lalu panggil compute_gate_only pada input yang SAMA.
    Returns: (dec_forward atau None, n_active_forward, dec_gate_only, prob_gate_only)
    """
    rt = find_router(model)
    cap = {}
    h = rt.register_forward_pre_hook(lambda m, inp: cap.update(p3=inp[0][0], p2=inp[0][1]))
    try:
        model(img)
    finally:
        h.remove()
    B = img.shape[0]
    dec_fwd = getattr(rt, "_last_gate_decision", None)        # butuh patch P2
    dec_fwd = dec_fwd.float().flatten().clone() if dec_fwd is not None else None
    n_fwd = float(rt.current_activation_prob) * B
    dec_only = rt.compute_gate_only(cap["p3"], cap["p2"]).float().flatten()
    # probabilitas (untuk margin ke threshold) — rumus sama dgn compute_gate_only
    z = rt._topk_spatial_pool(rt.squeeze_p3(cap["p3"]) + rt.squeeze_p2(cap["p2"]),
                              k=getattr(rt, "gate_topk", 10), alpha=getattr(rt, "gate_topk_alpha", 0.7))
    l = 5.0 * torch.tanh(rt.classifier(z).view(B, 2).float() / 5.0)
    prob = torch.sigmoid((l[:, 1] - l[:, 0]) * getattr(rt, "calib_T", 1.0) + getattr(rt, "calib_b", 0.0))
    return dec_fwd, n_fwd, dec_only, prob


def test_gate_consistency(model, batches, calibs, report=None):
    """
    Uji 0.5. Untuk tiap kalibrasi & tiap batch: keputusan forward(eval) == compute_gate_only.
    `calibs`: list (nama, T, b). Model harus .eval(); presisi mengikuti dtype model.
    """
    r = report or Report()
    rt = find_router(model)
    saved = _calib(rt)
    model.eval()
    try:
        for name, T, b in calibs:
            rt.calib_T, rt.calib_b = T, b
            n_img = n_mis = 0
            n_fwd_tot = n_only_tot = 0.0
            min_margin = 1.0
            weak = False
            for img in batches:
                dec_fwd, n_fwd, dec_only, prob = gate_decisions_both_paths(model, img)
                n_img += img.shape[0]
                n_fwd_tot += n_fwd
                n_only_tot += float(dec_only.sum())
                min_margin = min(min_margin, float((prob - 0.5).abs().min()))
                if dec_fwd is None:
                    weak = True
                else:
                    n_mis += int((dec_fwd.cpu() != dec_only.cpu()).sum())
            if weak:
                r.check(f"Gate [{name}] jumlah aktif forward == compute_gate_only (VERSI LEMAH, patch P2 belum)",
                        abs(n_fwd_tot - n_only_tot) < 0.5,
                        f"{n_fwd_tot:.0f} vs {n_only_tot:.0f} dari {n_img}", warn_only=True)
            else:
                r.check(f"Gate [{name}] keputusan per-gambar identik", n_mis == 0,
                        f"{n_mis}/{n_img} beda | aktif {n_only_tot:.0f}/{n_img} "
                        f"({n_only_tot / max(n_img, 1):.1%}) | margin min |p−0.5| = {min_margin:.2e}")
    finally:
        rt.calib_T, rt.calib_b = saved
    return r


def percentile_bias_for_rate(model, batches, target_rate, T=1.0):
    """b sehingga fraksi (logit·T + b > 0) ≈ target_rate pada `batches` (gaya DynamicDet).
    Dipakai HANYA untuk membuat kalibrasi uji yang menaruh banyak sampel dekat threshold."""
    rt = find_router(model)
    saved = _calib(rt)
    rt.calib_T, rt.calib_b = 1.0, 0.0
    logits = []
    for img in batches:
        _, _, _, prob = gate_decisions_both_paths(model, img)
        logits.append(torch.logit(prob.clamp(1e-7, 1 - 1e-7)).cpu())
    rt.calib_T, rt.calib_b = saved
    lg = torch.cat(logits) * T
    return float(-torch.quantile(lg, 1.0 - target_rate))




# =============================================================================
# PATCH SUMBER (EOL-aware, idempotent) — dipakai notebook sebelum import ultralytics
# =============================================================================
def apply_source_patch(path, old, new, done_marker, name):
    """
    Ganti `old` -> `new` tepat satu kali di file `path`, mempertahankan EOL file (LF/CRLF).
    Idempotent: bila `done_marker` sudah ada, file tidak diubah. Returns status string.
    """
    path = Path(path)
    raw = path.read_bytes().decode("utf-8")
    if done_marker in raw:
        return f"ℹ️ {name}: sudah ada — tidak di-patch."
    if "\r\n" in raw:
        old, new = old.replace("\n", "\r\n"), new.replace("\n", "\r\n")
    n = raw.count(old)
    if n != 1:
        raise RuntimeError(f"{name}: pola ditemukan {n}x (harus 1) di {path} — versi file berbeda, cek manual.")
    path.write_bytes(raw.replace(old, new).encode("utf-8"))
    return f"✅ {name}: di-patch."


PATCH_P2 = dict(
    name="P2 keputusan & logit gate per-gambar (block.py, eval)",
    rel="ultralytics/nn/modules/block.py",
    done_marker="self._last_gate_logit_raw = raw_logit_diff.detach()",
    old=("            if not torch.jit.is_tracing():\n"
         "                self.current_activation_prob = gate_mask.mean().detach()\n\n"
         "            f_c2f = self.compute_expert(f_p3, f_p2_back)\n"),
    new=("            if not torch.jit.is_tracing():\n"
         "                self.current_activation_prob = gate_mask.mean().detach()\n"
         "                # [TAHAP0] per-gambar: keputusan (uji konsistensi vs compute_gate_only) dan logit\n"
         "                # mentah sebelum kalibrasi (sebaran logit di log validasi)\n"
         "                self._last_gate_decision = gate_mask.view(B).detach()\n"
         "                self._last_gate_logit_raw = raw_logit_diff.detach()\n\n"
         "            f_c2f = self.compute_expert(f_p3, f_p2_back)\n"),
)

PATCH_CONF_SIGN = dict(
    name="Tanda conf di difficulty score (loss.py): - menjadi +",
    rel="ultralytics/utils/loss.py",
    done_marker="diff_score = entropy_f + var_f + conf_f",
    old=("                # TODO: verifikasi -- router.last_conf adalah UNCERTAINTY (1 - prob, tinggi =\n"
         "                # tidak yakin), sama arah dengan entropy dan var. Mengurangkannya di sini\n"
         "                # berarti ketidakpastian kelas yang lebih tinggi MENURUNKAN skor kesulitan.\n"
         "                # Apakah tanda minus ini disengaja (mis. dulu conf berarti keyakinan)?\n"
         "                diff_score = entropy_f + var_f - conf_f\n"),
    new=("                # [FIX 2026-10-09] router.last_conf adalah UNCERTAINTY (1 - prob, tinggi = tidak\n"
         "                # yakin), searah dengan entropy dan var. Tanda minus versi lama adalah salah ketik\n"
         "                # (dikonfirmasi penulis): ketiga sinyal ketidakpastian kini DIJUMLAHKAN.\n"
         "                # Run sebelum fix ini memakai entropy + var - conf (tidak sebanding langsung).\n"
         "                diff_score = entropy_f + var_f + conf_f\n"),
)


# =============================================================================
# GO/NO-GO KUALITAS RANKING GATE (mis. checkpoint epoch 50, sebelum lambda naik)
# =============================================================================
def _raw_gate_logit(rt, p3, p2):
    """Logit gate mentah (sebelum kalibrasi) — rumus identik dengan compute_gate_only."""
    z = rt._topk_spatial_pool(rt.squeeze_p3(p3) + rt.squeeze_p2(p2),
                              k=getattr(rt, "gate_topk", 10), alpha=getattr(rt, "gate_topk_alpha", 0.7))
    l = 5.0 * torch.tanh(rt.classifier(z).view(p3.shape[0], 2).float() / 5.0)
    return l[:, 1] - l[:, 0]


def _auc_with_ci(labels, scores, n_boot=1000, seed=0):
    """AUC (Mann-Whitney, ties = 0.5) + CI 95% bootstrap persentil. labels: 1 = Branch A lebih baik."""
    from scipy.stats import rankdata
    labels = np.asarray(labels, bool); scores = np.asarray(scores, float)

    def auc(lb, sc):
        n1, n0 = lb.sum(), (~lb).sum()
        if n1 == 0 or n0 == 0:
            return np.nan
        r = rankdata(sc)
        return (r[lb].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

    a = auc(labels, scores)
    rng = np.random.default_rng(seed); n = len(labels); boots = []
    for _ in range(n_boot):
        i = rng.integers(0, n, n)
        boots.append(auc(labels[i], scores[i]))
    boots = np.array([b for b in boots if np.isfinite(b)])
    lo, hi = (np.percentile(boots, [2.5, 97.5]) if len(boots) else (np.nan, np.nan))
    return float(a), float(lo), float(hi)


@torch.no_grad()
def gate_ranking_report(model, loader, device, max_batches=None, signal_mode="small",
                        min_std=0.05, n_boot=1000):
    """
    Apakah gate sudah MERANKING gambar sesuai sinyal yang melatihnya? (model beku, FP32, eval)

    Per gambar val: logit gate mentah + loss_A/loss_B per-sample (total & small, dari criterion
    yang sama dengan training). Oracle = 1 bila loss_A < loss_B. Untuk mode 'small' hanya
    gambar dengan objek kecil ter-assign di KEDUA branch yang dipakai (sama dgn valid_mask training).

    GO bila: std logit > min_std DAN batas bawah CI 95% AUC (mode sinyal training) > 0.5.
    Catatan: oracle memakai sinyal loss yang noisy — AUC menjawab "apakah gate mempelajari
    sinyal yang diberikan", bukan "apakah keputusan itu benar secara semantik".
    """
    from ultralytics.cfg import get_cfg
    from ultralytics.utils import DEFAULT_CFG
    model = deepcopy(model).float().eval().to(device)
    if isinstance(getattr(model, "args", None), dict) or not hasattr(model, "args"):
        a = getattr(model, "args", {}) or {}
        model.args = get_cfg(DEFAULT_CFG, overrides={k: a[k] for k in ("box", "cls", "dfl") if k in a})
    rt = find_router(model)
    crit = model.init_criterion()
    feats, rows = {}, {k: [] for k in ("raw", "la", "lb", "las", "lbs", "nsa", "nsb")}
    h = rt.register_forward_pre_hook(lambda m, inp: feats.update(p3=inp[0][0], p2=inp[0][1]))
    try:
        for bi, batch in enumerate(loader):
            if max_batches is not None and bi >= max_batches:
                break
            img = batch["img"].to(device).float() / 255.0
            b = {"img": img, **{k: batch[k].to(device) for k in ("cls", "bboxes", "batch_idx")}}
            det_A, det_B = model._predict_once_dual(img)
            crit.loss_A(det_A, b); crit.loss_B(det_B, b)
            rows["raw"].append(_raw_gate_logit(rt, feats["p3"], feats["p2"]).cpu())
            rows["la"].append(crit.loss_A._last_per_sample_loss.float().cpu())
            rows["lb"].append(crit.loss_B._last_per_sample_loss.float().cpu())
            rows["las"].append(crit.loss_A._last_per_sample_loss_small.float().cpu())
            rows["lbs"].append(crit.loss_B._last_per_sample_loss_small.float().cpu())
            rows["nsa"].append(crit.loss_A._last_small_obj_count.float().cpu())
            rows["nsb"].append(crit.loss_B._last_small_obj_count.float().cpu())
    finally:
        h.remove()
    R = {k: torch.cat(v).numpy() for k, v in rows.items()}
    raw = R["raw"]
    prob = 1.0 / (1.0 + np.exp(-raw))
    out = dict(n_images=int(len(raw)), logit_mean=float(raw.mean()), logit_std=float(raw.std()),
               prob_std_uncalibrated=float(prob.std()),
               logit_p10=float(np.percentile(raw, 10)), logit_p90=float(np.percentile(raw, 90)),
               frac_saturated=float((np.abs(raw) > 9.5).mean()),
               activation_uncalibrated=float((raw > 0).mean()))
    a, lo, hi = _auc_with_ci(R["la"] < R["lb"], raw, n_boot)
    out.update(total=dict(n=int(len(raw)), oracle_rate=float((R["la"] < R["lb"]).mean()), auc=a, auc_ci=[lo, hi]))
    v = (R["nsa"] > 0) & (R["nsb"] > 0)
    if v.sum() >= 20:
        a, lo, hi = _auc_with_ci(R["las"][v] < R["lbs"][v], raw[v], n_boot)
        out.update(small=dict(n=int(v.sum()), oracle_rate=float((R["las"][v] < R["lbs"][v]).mean()),
                              auc=a, auc_ci=[lo, hi]))
    key = signal_mode if signal_mode in out else "total"
    out["decision_signal"] = key
    out["go"] = bool(out["logit_std"] > min_std and out[key]["auc_ci"][0] > 0.5)
    out["reason"] = (f"std={out['logit_std']:.4f} (> {min_std}?), AUC[{key}]={out[key]['auc']:.3f} "
                     f"CI95=[{out[key]['auc_ci'][0]:.3f}, {out[key]['auc_ci'][1]:.3f}] (batas bawah > 0.5?)")
    return out


def stop_training(trainer, reason):
    """
    Hentikan training dengan rapi setelah epoch ini. final_eval() -> strip_optimizer() membuat
    last.pt TIDAK bisa di-resume (epoch=-1, optimizer dibuang), jadi salinan utuh disimpan SEKARANG
    (save_model sudah jalan untuk epoch ini). Returns path salinan.
    """
    epoch = trainer.epoch + 1
    keep = Path(trainer.wdir) / f"last_resumable_epoch{epoch}.pt"
    # final_eval() memicu on_fit_epoch_end sekali lagi SETELAH last.pt di-strip -> jangan timpa salinan
    if getattr(trainer, "stop", False) or (Path(trainer.save_dir) / "STOPPED.txt").exists():
        return str(keep)
    if Path(trainer.last).exists():
        shutil.copy(trainer.last, keep)
    (Path(trainer.save_dir) / "STOPPED.txt").write_text(f"epoch {epoch}: {reason}\nresume: {keep}\n")
    print(f"⛔ Training dihentikan di epoch {epoch}: {reason}. Untuk lanjut: resume dari {keep}")
    trainer.stop = True
    return str(keep)


def make_activation_stop_rule(gate_val_csv, after_epoch=60, low=0.05, high=0.95, patience=5):
    """
    Aturan berhenti Tahap 1: aktivasi val (EMA, tanpa kalibrasi, dari gate_val_stats.csv) < low atau
    > high selama `patience` epoch berturut-turut SETELAH `after_epoch`. Dipasang SESUDAH callback yang
    menulis gate_val_stats.csv (on_val_end -> sudah tertulis saat on_fit_epoch_end).
    """
    def cb(trainer):
        epoch = trainer.epoch + 1
        if getattr(trainer, "stop", False) or epoch <= after_epoch or not Path(gate_val_csv).exists():
            return
        gv = pd.read_csv(gate_val_csv).drop_duplicates("epoch", keep="first").set_index("epoch")
        win = [e for e in range(epoch - patience + 1, epoch + 1)]
        if all(e in gv.index and e > after_epoch for e in win):
            a = gv.loc[win, "act_uncalibrated"]
            if (a < low).all() or (a > high).all():
                stop_training(trainer, f"aktivasi val {a.round(3).tolist()} di luar [{low}, {high}] "
                                       f"selama {patience} epoch (epoch {win[0]}–{win[-1]})")
    return cb


def make_logit_health_stop_rule(gate_val_csv, after_epoch=10, min_std=0.05, max_sat=0.20, patience=2):
    """
    Aturan berhenti Tahap 1 versi alpha = 0 (revisi 2026-10-09, menggantikan aturan aktivasi).
    Dengan alpha = 0 aktivasi deployment dipilih post-hoc lewat persentil, jadi yang harus dijaga
    adalah PERINGKAT logit: berhenti bila std logit val < min_std ATAU fraksi logit tersaturasi
    (|z| > 9.5, batas tanh +-10) > max_sat selama `patience` epoch berturut-turut setelah after_epoch
    (setelah warmup router: selama warmup logit memang hampir konstan).
    """
    def cb(trainer):
        epoch = trainer.epoch + 1
        if getattr(trainer, "stop", False) or epoch <= after_epoch or not Path(gate_val_csv).exists():
            return
        gv = pd.read_csv(gate_val_csv).drop_duplicates("epoch", keep="last").set_index("epoch")
        if "logit_std" not in gv:
            return
        win = list(range(epoch - patience + 1, epoch + 1))
        if all(e in gv.index and e > after_epoch for e in win):
            w = gv.loc[win]
            bad = (w.logit_std < min_std) | (w.frac_saturated > max_sat)
            if bad.all():
                stop_training(trainer, f"kesehatan logit gagal epoch {win[0]}–{win[-1]}: std={w.logit_std.round(4).tolist()} "
                                       f"(min {min_std}), saturasi={w.frac_saturated.round(3).tolist()} (maks {max_sat})")
    return cb


def make_auc_stop_rule(trainer_ref, out_csv, every=10, after_epoch=30, patience=2, signal_mode="total",
                       max_batches=None):
    """
    Pengaman Tahap 1 revisi ke-2 (2026-10-09, setelah run M0 seed 42 λ=0.04: AUC epoch 100 = 0.417,
    terbalik). Tiap `every` epoch: gate_ranking_report pada model EMA di val -> baris ke out_csv
    (epoch, AUC, CI, std logit, aktivasi). Berhenti bila batas bawah CI 95% AUC <= 0.5 pada
    `patience` pemeriksaan berturut-turut dengan epoch > after_epoch. Pemeriksaan <= after_epoch
    tetap dicatat (informatif). Biaya: +-1 menit per pemeriksaan di T4.
    """
    def cb(trainer):
        epoch = trainer.epoch + 1
        # Catatan: JANGAN pakai trainer.stop sebagai guard — Ultralytics men-set stop=True di epoch terakhir
        # sebelum on_fit_epoch_end, sehingga pemeriksaan epoch 100 terlewat (bug di run M0_lam002_seed42).
        if epoch % every != 0 or (Path(trainer.save_dir) / "STOPPED.txt").exists():
            return
        hist = pd.read_csv(out_csv) if Path(out_csv).exists() else pd.DataFrame()
        if not hist.empty and epoch in set(hist.epoch):
            return                                   # final_eval memicu callback lagi -> jangan dobel
        device = next(trainer.model.parameters()).device
        rep = gate_ranking_report(trainer.ema.ema, trainer.test_loader, device, max_batches=max_batches,
                                  signal_mode=signal_mode)
        key = rep["decision_signal"]
        row = dict(epoch=epoch, auc=rep[key]["auc"], auc_lo=rep[key]["auc_ci"][0], auc_hi=rep[key]["auc_ci"][1],
                   logit_std=rep["logit_std"], logit_mean=rep["logit_mean"],
                   act_uncalibrated=rep["activation_uncalibrated"],
                   auc_small=rep.get("small", {}).get("auc", np.nan))
        hist = pd.concat([hist, pd.DataFrame([row])], ignore_index=True)
        hist.to_csv(out_csv, index=False)
        print(f"\n📊 [AUC CHECK epoch {epoch}] AUC[{key}]={row['auc']:.3f} CI95=[{row['auc_lo']:.3f}, {row['auc_hi']:.3f}] "
              f"| std logit={row['logit_std']:.3f} | aktivasi={row['act_uncalibrated']:.3f}\n")
        late = hist[hist.epoch > after_epoch].sort_values("epoch").tail(patience)
        if len(late) == patience and (late.auc_lo <= 0.5).all():
            stop_training(trainer, f"ranking gate gagal: batas bawah CI AUC <= 0.5 pada epoch {late.epoch.tolist()} "
                                   f"(AUC {late.auc.round(3).tolist()})")
    return cb


def make_criterion_cfg_applier(criterion_cfg, dump_path=None):
    """
    on_pretrain_routine_end (SETELAH router cfg): buat criterion sekarang dan set atribut yang dibaca
    v8DetectionLoss dari DIRINYA SENDIRI (mis. normalize_bg_anchors), bukan dari model.
    Model.loss() hanya membuat criterion bila belum ada, jadi criterion ini yang dipakai training.
    """
    def apply(trainer):
        m = trainer.model
        m.criterion = m.init_criterion()
        for k, v in criterion_cfg.items():
            setattr(m.criterion.loss_A, k, v); setattr(m.criterion.loss_B, k, v)
        if dump_path is not None:
            Path(dump_path).write_text(json.dumps(criterion_cfg, indent=1))
        print("🔧 [CRITERION CFG]", criterion_cfg)
    return apply


def make_gate_ranking_check(trainer_ref, at_epoch, out_path, signal_mode="small", stop_on_nogo=True,
                            max_batches=None):
    """
    on_fit_epoch_end: di akhir epoch `at_epoch`, jalankan gate_ranking_report pada model EMA
    (val loader trainer). NO-GO + stop_on_nogo -> trainer.stop = True (training berhenti rapi,
    last.pt tersimpan, bisa dilanjutkan dengan resume=True bila diputuskan lanjut).
    """
    def cb(trainer):
        epoch = trainer.epoch + 1
        if epoch != at_epoch or Path(out_path).exists():
            return
        device = next(trainer.model.parameters()).device
        rep = gate_ranking_report(trainer.ema.ema, trainer.test_loader, device, max_batches=max_batches,
                                  signal_mode=signal_mode)
        rep.update(epoch=epoch, stop_on_nogo=stop_on_nogo)
        Path(out_path).write_text(json.dumps(rep, indent=1))
        print(f"\n🚦 [GATE CHECK epoch {epoch}] {'GO' if rep['go'] else 'NO-GO'} — {rep['reason']}\n")
        if not rep["go"] and stop_on_nogo:
            rep["resumable_checkpoint"] = stop_training(trainer, "gate check NO-GO")
            Path(out_path).write_text(json.dumps(rep, indent=1))
    return cb

In [ ]:
%%writefile /kaggle/working/thesis-Difficulty-Aware-Router/script_training.py
# [5] versi sama dengan Tahap 1 (hapus sel ini setelah file di-commit ke repo)
"""
script_training.py — Training Router-P2 YOLOv8n Dual-Branch (Skenario 3), versi importable.

Perubahan dibanding kode_berkomentar/script_training.py (logika model/loss TIDAK berubah):
  - Section 1 lama (patch loss.py di disk lewat penggantian string) DIHAPUS: guard
    _compute_router_penalty dan default lambda sudah ada di loss.py, sehingga patch itu
    tidak pernah mengubah apa pun, dan menulis file library saat runtime berisiko.
  - Eksekusi dibungkus build_trainer() + main(); import file ini tidak memulai training.
  - ideal_curriculum_scheduler diganti make_curriculum_scheduler (router_train_utils),
    yang dengan FULL_SCHED identik dengan jadwal lama (diuji epoch 1–100) kecuali lambda
    final = 0.04 (keputusan 2026-10-09).
  - Hyperparameter router ditulis eksplisit (ROUTER_CFG); router_hybrid_alpha = 0.
  - Semua log ke save_dir run: router_loss_components.csv (11 kolom, label epoch benar),
    loss_items.csv, epoch_timing.csv, branch_b_metrics.csv, router_cfg.json, sched_history.json.
  - Penjaga NaN per batch (gagal cepat, bukan NaN recovery diam-diam Ultralytics).
  - Sebaran logit gate per validasi (gate_val_stats.csv) dan go/no-go ranking gate di akhir
    epoch lam_start (50), tepat sebelum lambda naik (gate_check.json; NO-GO -> berhenti).
  - Butuh patch sumber (notebook sel 4): P2 logit/keputusan gate per-gambar, tanda conf (+).
"""
import os
import csv
import json
from copy import copy
from pathlib import Path

import pandas as pd
import torch
from ultralytics import settings
from ultralytics.models.yolo.detect import DetectionTrainer, DetectionValidator
from ultralytics.model_yolov8_router import DualBranchDetectionModel

from router_train_utils import (EpochTimer, make_activation_stop_rule, make_criterion_cfg_applier,
                                make_logit_health_stop_rule, make_auc_stop_rule,
                                make_curriculum_scheduler, make_gate_ranking_check, make_loss_items_logger,
                                make_router_cfg_applier, make_router_csv_logger)

# ==============================================================================
# KONFIGURASI RUN UTAMA
# ==============================================================================
BASE_OVERRIDES = dict(
    model="ultralytics/cfg/models/v8/yolov8n-p2-router.yaml",
    data="/kaggle/input/datasets/agungmaugi/kitti-2class/kitti_2class.yaml",
    batch=16, imgsz=640, optimizer="AdamW", lr0=0.002, cos_lr=True, weight_decay=0.0005,
    device=0, workers=4, seed=42, amp=True, verbose=True, save=True,
)
# save_period=10: epoch50.pt (go/no-go) + titik resume bila sesi Kaggle putus.
FULL_OVERRIDES = {**BASE_OVERRIDES, "epochs": 100, "project": "/kaggle/working/runs_tahap1",
                  "name": "M0_lam002_seed42", "save_period": 10}

# Fase 1 (1–50): lambda 0; router warmup 1–10; gate_value_weight 0 -> 0.5 di epoch 11–30.
# Fase 2 (51–80): lambda 0 -> 0.02 linear. Fase 3 (81–100): lambda 0.02.
# Fase 2 (51–80): lambda 0 -> final linear. Fase 3 (81–100): lambda final.
FULL_SCHED = dict(router_warmup_end=10, gv_start=10, gv_ramp=20, final_gv=0.5,
                  lam_start=50, lam_ramp=30, final_lambda=0.02)
# λ akhir 0.02 (keputusan 2026-10-09 malam). Run pertama λ=0.04 (seed 42) berakhir dengan ranking gate
# terbalik (AUC 0.417 di epoch 100) dan kini dilaporkan sebagai ablasi.

# Keputusan 2026-10-09: alpha = 0 -> router_target_activation tidak berpengaruh; sparsity bekerja
# sebagai harga P2 (c = lambda * Diff_W / gate_value_weight); activation rate deployment dipilih
# post-hoc (operating_point_sweep.py). Nilai target tetap ditulis supaya konfigurasi eksplisit.
# gate_signal_mode='total' (rencana, konfigurasi terkunci): 'small' membuang gambar yang objek kecilnya
# hanya tertangkap P2 (bias seleksi); 'small' diuji sebagai ablasi A5.
ROUTER_CFG = dict(
    gate_signal_mode="total",
    router_hybrid_alpha=0.0,
    router_target_activation=0.35,
    router_activation_tolerance=0.10,
    router_min_activation_floor=0.05,
    branch_b_loss_weight=0.7,
    gate_offset_momentum=0.99,
)
# Pengaman revisi ke-2 (2026-10-09): AUC ranking gate di val tiap 10 epoch; berhenti bila batas bawah
# CI <= 0.5 dua pemeriksaan berturut-turut setelah epoch 30. Menggantikan aturan aktivasi (rencana awal)
# dan aturan std/saturasi logit (revisi ke-1), yang tidak menangkap ranking terbalik di run λ=0.04.
AUC_STOP_RULE = dict(every=10, after_epoch=30, patience=2)

# Atribut yang dibaca v8DetectionLoss dari dirinya sendiri (bukan dari model) -> diset ke criterion.
CRITERION_CFG = dict(normalize_bg_anchors=True)   # ablasi A6: False


# ==============================================================================
# TRAINER & VALIDATOR
# ==============================================================================
class DualBranchTrainer(DetectionTrainer):
    """DetectionTrainer yang membangun DualBranchDetectionModel dan 8 nama komponen loss."""

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = DualBranchDetectionModel(cfg=cfg or self.args.model, ch=3, nc=self.data["nc"], verbose=verbose)
        if weights:
            model.load(weights)
        return model

    def get_validator(self):
        # Urutan = combined_items di DualBranchDetectionLoss: 5 item loss_A + 3 item loss_B.
        self.loss_names = ("box_A", "cls_A", "dfl_A", "router", "proxy", "box_B", "cls_B", "dfl_B")
        return DetectionValidator(self.test_loader, save_dir=self.save_dir, args=copy(self.args),
                                  _callbacks=self.callbacks)


class DualBranchValidatorB(DetectionValidator):
    """Validator yang mengevaluasi Branch B: _predict_once diganti sementara, selalu dipulihkan."""

    def __call__(self, trainer=None, model=None):
        if model is not None:
            target_model = model
        elif trainer is not None:
            target_model = trainer.ema.ema if (hasattr(trainer, "ema") and trainer.ema) else trainer.model
        else:
            raise ValueError("DualBranchValidatorB butuh `trainer` atau `model`.")
        assert hasattr(target_model, "_predict_once_dual"), "Model bukan DualBranchDetectionModel."

        original = target_model._predict_once

        def branch_b_predict_once(x, profile=False, visualize=False, embed=None):
            _, det_B = target_model._predict_once_dual(x, profile, visualize, embed)
            return det_B

        target_model._predict_once = branch_b_predict_once
        try:
            return super().__call__(trainer=trainer, model=model)
        finally:
            target_model._predict_once = original


# ==============================================================================
# CALLBACK VALIDASI
# ==============================================================================
def make_router_stat_checker(trainer_ref, csv_path=None):
    """
    Aktivasi P2 + sebaran logit gate mentah di validasi, per epoch -> csv_path.
    Angka ini TANPA kalibrasi, FP16, dataloader rect -> bukan angka deployment; gunanya memantau
    apakah logit gate tersebar antar-gambar (syarat kalibrasi persentil) sepanjang training.
    Butuh patch P2 (router._last_gate_logit_raw); tanpa patch hanya aktivasi yang tercatat.
    """
    stats = {"total": 0.0, "n": 0, "logits": []}

    def on_val_start(validator):
        stats["total"], stats["n"], stats["logits"] = 0.0, 0, []

    def on_val_batch_end(validator):
        m_ = trainer_ref.ema.ema if (hasattr(trainer_ref, "ema") and trainer_ref.ema) else trainer_ref.model
        if m_ is None:
            return
        for m in m_.modules():
            if "DifficultyAwareRouter" in m.__class__.__name__:
                v = getattr(m, "current_activation_prob", None)
                if v is not None:
                    stats["total"] += v.item() if hasattr(v, "item") else float(v)
                    stats["n"] += 1
                lg = getattr(m, "_last_gate_logit_raw", None)
                if lg is not None:
                    stats["logits"].append(lg.detach().float().flatten().cpu())
                break

    def on_val_end(validator):
        if not stats["n"]:
            return
        act = stats["total"] / stats["n"]
        msg = f"P2 AKTIF: {act * 100:.2f}%"
        row = {"epoch": trainer_ref.epoch + 1, "act_uncalibrated": act}
        if stats["logits"]:
            lg = torch.cat(stats["logits"])
            row.update(n=int(lg.numel()), logit_mean=float(lg.mean()), logit_std=float(lg.std()),
                       logit_p10=float(lg.quantile(0.1)), logit_p90=float(lg.quantile(0.9)),
                       frac_saturated=float((lg.abs() > 9.5).float().mean()))
            msg += f" | logit mean={row['logit_mean']:.3f} std={row['logit_std']:.4f}"
        print(f"\n📉 [EVALUASI ROUTER] {msg}\n")
        if csv_path is not None:
            pd.DataFrame([row]).to_csv(csv_path, mode="a", header=not Path(csv_path).exists(), index=False)

    return on_val_start, on_val_batch_end, on_val_end


def make_branch_b_checker(trainer_ref, interval=10, csv_path=None):
    """Evaluasi Branch B tiap `interval` epoch; hasil ke csv_path (default: save_dir/branch_b_metrics.csv)."""
    def on_val_end_check_branch_b(validator):
        epoch = trainer_ref.epoch + 1
        if epoch % interval != 0:
            return
        path = Path(csv_path or Path(trainer_ref.save_dir) / "branch_b_metrics.csv")
        vb = DualBranchValidatorB(dataloader=trainer_ref.test_loader, save_dir=trainer_ref.save_dir,
                                  args=copy(trainer_ref.args))
        try:
            vb(trainer=trainer_ref)
            bm = vb.metrics.box
            print(f"🔍 [BRANCH B] Epoch {epoch} | mAP50: {bm.map50:.4f} | mAP50-95: {bm.map:.4f} | "
                  f"P: {bm.mp:.4f} | R: {bm.mr:.4f}")
            new = not path.exists()
            with open(path, "a", newline="") as f:
                w = csv.writer(f)
                if new:
                    w.writerow(["epoch", "mAP50", "mAP50-95", "precision", "recall"])
                w.writerow([epoch, bm.map50, bm.map, bm.mp, bm.mr])
        except Exception as e:  # jangan hentikan training karena evaluasi tambahan
            import traceback
            print(f"⚠️  [BRANCH B CHECK GAGAL] {type(e).__name__}: {e}")
            traceback.print_exc()

    return on_val_end_check_branch_b


# ==============================================================================
# BANGUN TRAINER (dipakai run utama DAN smoke test)
# ==============================================================================
def build_trainer(overrides, sched, router_cfg=ROUTER_CFG, branch_b_interval=10,
                  gate_check_epoch="auto", stop_on_nogo=True, criterion_cfg=CRITERION_CFG,
                  activation_stop_rule=None, health_stop_rule=None, auc_stop_rule=None):
    """
    Returns (trainer, ctx). ctx berisi objek log yang bisa diperiksa setelah train():
      timer, sched_history, loss_items_cb, paths (semua di trainer.save_dir).
    Catatan: save_dir sudah ditentukan di konstruktor trainer, jadi path log aman dipakai.

    gate_check_epoch: epoch go/no-go ranking gate ("auto" = sched["lam_start"], tepat sebelum
    lambda naik; None = mati). NO-GO + stop_on_nogo -> training berhenti setelah epoch itu.
    activation_stop_rule: dict untuk make_activation_stop_rule (aturan rencana lama, untuk alpha > 0), None = mati.
    health_stop_rule: dict untuk make_logit_health_stop_rule (revisi ke-1, terbukti terlalu longgar), None = mati.
    auc_stop_rule: dict untuk make_auc_stop_rule (revisi ke-2: AUC ranking tiap N epoch), None = mati.
    """
    settings.update({"raytune": False})
    resuming = bool(overrides.get("resume"))
    trainer = DualBranchTrainer(overrides=overrides)
    sd = Path(trainer.save_dir); sd.mkdir(parents=True, exist_ok=True)
    paths = dict(router_csv=sd / "router_loss_components.csv", loss_items=sd / "loss_items.csv",
                 timing=sd / ("epoch_timing_resume.csv" if resuming else "epoch_timing.csv"), branch_b=sd / "branch_b_metrics.csv",
                 router_cfg=sd / "router_cfg.json", sched=sd / "sched_history.json",
                 gate_val=sd / "gate_val_stats.csv", gate_check=sd / "gate_check.json",
                 criterion_cfg=sd / "criterion_cfg.json", auc_monitor=sd / "auc_monitor.csv")

    sched_history, timer = {}, EpochTimer()
    li_store = []
    li_cb = make_loss_items_logger(li_store, paths["loss_items"])
    r_start, r_batch, r_end = make_router_stat_checker(trainer, paths["gate_val"])
    if gate_check_epoch == "auto":
        gate_check_epoch = sched["lam_start"]

    def save_epoch_logs(tr):
        pd.DataFrame(timer.rows).to_csv(paths["timing"], index=False)
        paths["sched"].write_text(json.dumps({str(k): v for k, v in sched_history.items()}, indent=1))

    trainer.add_callback("on_pretrain_routine_end", make_router_cfg_applier(router_cfg, paths["router_cfg"]))
    trainer.add_callback("on_pretrain_routine_end", make_criterion_cfg_applier(criterion_cfg, paths["criterion_cfg"]))
    trainer.add_callback("on_train_epoch_start", make_curriculum_scheduler(**sched, history=sched_history))
    trainer.add_callback("on_train_epoch_start", timer.on_train_epoch_start)
    trainer.add_callback("on_train_batch_end", li_cb)
    trainer.add_callback("on_train_epoch_end", timer.on_train_epoch_end)          # sebelum flush CSV
    trainer.add_callback("on_train_epoch_end", make_router_csv_logger(paths["router_csv"], allow_existing=resuming))
    trainer.add_callback("on_train_epoch_end", li_cb.flush)
    trainer.add_callback("on_val_start", r_start)
    trainer.add_callback("on_val_batch_end", r_batch)
    trainer.add_callback("on_val_end", r_end)
    trainer.add_callback("on_val_end", timer.timed(make_branch_b_checker(trainer, branch_b_interval, paths["branch_b"])))
    trainer.add_callback("on_fit_epoch_end", timer.on_fit_epoch_end)
    trainer.add_callback("on_fit_epoch_end", save_epoch_logs)
    if gate_check_epoch:
        trainer.add_callback("on_fit_epoch_end", make_gate_ranking_check(
            trainer, gate_check_epoch, paths["gate_check"], signal_mode=router_cfg.get("gate_signal_mode", "small"),
            stop_on_nogo=stop_on_nogo))
    if activation_stop_rule:
        trainer.add_callback("on_fit_epoch_end", make_activation_stop_rule(paths["gate_val"], **activation_stop_rule))
    if health_stop_rule:
        trainer.add_callback("on_fit_epoch_end", make_logit_health_stop_rule(paths["gate_val"], **health_stop_rule))
    if auc_stop_rule:
        trainer.add_callback("on_fit_epoch_end", make_auc_stop_rule(
            trainer, paths["auc_monitor"], signal_mode=router_cfg.get("gate_signal_mode", "total"), **auc_stop_rule))

    ctx = dict(timer=timer, sched_history=sched_history, loss_items_cb=li_cb, paths=paths,
               sched=sched, router_cfg=router_cfg, criterion_cfg=criterion_cfg, overrides=overrides,
               gate_check_epoch=gate_check_epoch)
    return trainer, ctx


def main():
    print("🔥 SKENARIO 3: Router-P2 YOLOv8n Dual-Branch (Difficulty-Aware Router)")
    trainer, ctx = build_trainer(FULL_OVERRIDES, FULL_SCHED, ROUTER_CFG, branch_b_interval=10,
                                 auc_stop_rule=AUC_STOP_RULE)
    trainer.train()
    return trainer, ctx


if __name__ == "__main__":
    main()

In [ ]:
# [6] PATCH SUMBER (idempotent, EOL-aware) — hapus sel ini setelah kedua patch di-commit ke repo
#   P2   block.py : simpan keputusan & logit gate mentah per-gambar di forward eval (uji 0.5 + log validasi).
#                   Di dalam blok `not torch.jit.is_tracing()` -> graf export tidak berubah.
#   CONF loss.py  : difficulty score = entropy + var + conf (tanda minus lama = salah ketik, 2026-10-09).
sys.path.insert(0, str(REPO))
import router_train_utils as U
for P_ in (U.PATCH_P2, U.PATCH_CONF_SIGN):
    print(U.apply_source_patch(REPO / P_["rel"], P_["old"], P_["new"], P_["done_marker"], P_["name"]))
PROV["git_diff_sha256"] = hashlib.sha256(sh(["git", "diff"], cwd=REPO).encode()).hexdigest()
PROV["git_status"] = sh(["git", "status", "--short"], cwd=REPO)
print("git status:\n" + PROV["git_status"])

In [ ]:
# [7] VERIFIKASI VERSI KODE — berhenti di sini bila sesi ini tidak memakai kode yang dimaksud
import inspect
sys.path.insert(0, str(REPO)); os.chdir(REPO)
import torch, ultralytics
import ultralytics.nn.modules.block as _blk
import ultralytics.model_yolov8_router as _mr
import ultralytics.utils.loss as _ls
_src_r = inspect.getsource(_blk.UltraLightWeightDifficultyAwareRouter)
CODE_FEATURES = {
    "ultralytics di-import dari repo (bukan pip)": Path(ultralytics.__file__).resolve().is_relative_to(REPO.resolve()),
    "AUDIT-FIX #9 kalibrasi = buffer int64": "_calib_T_micro" in _src_r,
    "GroupNorm di classifier gate": "GroupNorm(1, hidden_dim // 2)" in _src_r,
    "top-k spatial pool di gate": "_topk_spatial_pool" in _src_r,
    "patch P2 keputusan + logit gate per-gambar": "_last_gate_logit_raw" in _src_r,
    "fix tanda conf (+) di difficulty score": "entropy_f + var_f + conf_f" in inspect.getsource(_ls.v8DetectionLoss),
    "AUDIT-FIX #6 kolom GV tanpa lag": hasattr(_mr.DualBranchDetectionLoss, "_flush_pending_log"),
    "AUDIT-FIX #3 advantage di-clip": "gate_adv_clip" in inspect.getsource(_mr.DualBranchDetectionLoss),
    "AUDIT-FIX #7 normalisasi bg anchor": "normalize_bg_anchors" in inspect.getsource(_ls.v8DetectionLoss),
}
for k, v in CODE_FEATURES.items():
    print(("✅ " if v else "❌ ") + k)
PROV.update(torch=torch.__version__, ultralytics=ultralytics.__version__, python=platform.python_version(),
            cuda=torch.version.cuda, gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
            code_features=CODE_FEATURES)
print({k: PROV[k] for k in ("torch", "ultralytics", "cuda", "gpu")})
assert all(CODE_FEATURES.values()), "Versi kode BUKAN yang dimaksud — push/clone ulang, lalu Restart & Run All."
assert torch.cuda.is_available() or "CFG_OVERRIDE" in globals(), "GPU tidak aktif (Settings → Accelerator → GPU T4)."

In [ ]:
%%writefile /kaggle/working/thesis-Difficulty-Aware-Router/tahap2_eval.py
# [8] modul evaluasi Tahap 2 (hapus sel ini setelah file di-commit ke repo)
"""
tahap2_eval.py — Tahap 2: kalibrasi & protokol evaluasi Router-P2 Dual-Branch (satu checkpoint per run).

Isi (dipanggil dari notebook tahap2_eval.ipynb):
  - Muat checkpoint (state_dict -> model baru, kalibrasi persisten), stage 1/2A/2B (sama dgn deployment)
  - cache_split(): SATU forward per gambar -> logit gate mentah z, output Branch A & B, loss per-sample,
    info GT. Semua mode evaluasi (A paksa, B paksa, ter-route pada threshold apa pun, acak, oracle)
    dirakit dari cache yang sama -> NMS/validator identik, biaya murah.
  - Sweep titik operasi (router vs acak) di val -> r*; kalibrasi persentil (T=1, b=-q).
  - COCO (pycocotools) + bootstrap berpasangan per gambar (AP, AP50, AP_small) dari evalImgs
    tanpa menjalankan ulang COCOeval.evaluate().
  - Perilaku gate (RQ3) dgn z (peringkat), koreksi Holm; diagnostik BatchNorm gate (close_mosaic).
"""
import json
import sys
import time
import copy
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from router_train_utils import find_router, _raw_gate_logit, _auc_with_ci


# =============================================================================
# MODEL & STAGE
# =============================================================================
def model_fingerprint(m, skip_calib=True):
    """Hash state_dict. Buffer kalibrasi dikecualikan (berubah sah saat kalibrasi)."""
    import hashlib
    h = hashlib.sha256()
    for k, v in sorted(m.state_dict().items()):
        if skip_calib and "_calib_" in k:
            continue
        h.update(k.encode()); h.update(v.detach().float().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()


def set_calibration(router, T, b):
    router.calib_T = float(T); router.calib_b = float(b)


def get_calibration(router):
    return float(router.calib_T), float(router.calib_b)


def load_dualbranch(path, model_yaml, nc, names, device):
    """
    Muat checkpoint Ultralytics (last.pt ter-strip: 'model' = EMA) ke DualBranchDetectionModel baru
    lewat state_dict (strict=False hanya untuk buffer kalibrasi). Returns (model eval, info).
    """
    from ultralytics.model_yolov8_router import DualBranchDetectionModel
    from ultralytics.cfg import get_cfg
    from ultralytics.utils import DEFAULT_CFG
    ck = torch.load(path, map_location="cpu", weights_only=False)
    key = "ema" if ck.get("ema") is not None else "model"
    src = ck[key]
    sd = {k: (v.float() if v.is_floating_point() else v) for k, v in src.state_dict().items()}
    m = DualBranchDetectionModel(cfg=model_yaml, ch=3, nc=nc, verbose=False)
    missing, unexpected = m.load_state_dict(sd, strict=False)
    bad = [k for k in missing if "calib" not in k]
    assert not bad and not unexpected, f"State dict tidak cocok: missing={bad[:5]} unexpected={list(unexpected)[:5]}"
    hyp = ck.get("train_args") or {}
    m.args = get_cfg(DEFAULT_CFG, overrides={k: hyp[k] for k in ("box", "cls", "dfl") if k in hyp})
    m.names = names
    m = m.to(device).eval()
    for p in m.parameters():
        p.requires_grad_(False)
    info = dict(ckpt=str(path), weights_key=key, epoch=ck.get("epoch"), train_args_name=hyp.get("name"),
                train_args_seed=hyp.get("seed"), calib_in_ckpt=get_calibration(find_router(m)))
    return m, info


class Stage1_BackboneNeckGate(nn.Module):
    """Layer 0-15 + gate (compute_gate_only: kalibrasi + threshold)."""
    def __init__(self, full_model):
        super().__init__()
        ml = list(full_model.model)
        self.pre_router_layers = nn.ModuleList(ml[0:16])
        self.router = ml[16]
        assert "DifficultyAwareRouter" in type(self.router).__name__

    def forward(self, x):
        y = []
        for m in self.pre_router_layers:
            if m.f != -1:
                x = y[m.f] if isinstance(m.f, int) else [x if j == -1 else y[j] for j in m.f]
            x = m(x)
            y.append(x)
        p3_neck, p2_backbone, p4_neck, p5_sppf = x, y[2], y[12], y[9]
        gate = self.router.compute_gate_only(p3_neck, p2_backbone)
        return gate, p3_neck, p2_backbone, p4_neck, p5_sppf


class Stage2A_WithP2(nn.Module):
    """Layer 16 (expert) + 17-25 + Detect_A -> y (B, 4+nc, N_A)."""
    def __init__(self, full_model):
        super().__init__()
        ml = list(full_model.model)
        self.router, self.branch_a_layers, self.detect_A = ml[16], nn.ModuleList(ml[17:26]), ml[32]

    def forward(self, p3_neck, p2_backbone, p4_neck, p5_sppf):
        p2_out = self.router.compute_expert_only(p3_neck, p2_backbone)
        y = {15: p3_neck, 12: p4_neck, 9: p5_sppf, 16: p2_out}
        x = p2_out
        for i, m in zip(range(17, 26), self.branch_a_layers):
            if m.f != -1:
                x = y[m.f] if isinstance(m.f, int) else [x if j == -1 else y[j] for j in m.f]
            x = m(x)
            y[i] = x
        out = self.detect_A([p2_out, y[19], y[22], y[25]])
        return out[0] if isinstance(out, tuple) else out


class Stage2B_NoP2(nn.Module):
    """Layer 26-31 + Detect_B -> y (B, 4+nc, N_B)."""
    def __init__(self, full_model):
        super().__init__()
        ml = list(full_model.model)
        self.branch_b_layers, self.detect_B = nn.ModuleList(ml[26:32]), ml[33]

    def forward(self, p3_neck, p4_neck, p5_sppf):
        y = {15: p3_neck, 12: p4_neck, 9: p5_sppf}
        x = p3_neck
        for i, m in zip(range(26, 32), self.branch_b_layers):
            if m.f != -1:
                x = y[m.f] if isinstance(m.f, int) else [x if j == -1 else y[j] for j in m.f]
            x = m(x)
            y[i] = x
        out = self.detect_B([p3_neck, y[28], y[31]])
        return out[0] if isinstance(out, tuple) else out


def make_stages(model):
    s1, s2a, s2b = Stage1_BackboneNeckGate(model).eval(), Stage2A_WithP2(model).eval(), Stage2B_NoP2(model).eval()
    return s1, s2a, s2b


@torch.no_grad()
def check_stage_equivalence(model, stages, loader, device, n=2, tol=1e-5):
    s1, s2a, s2b = stages
    for k, batch in enumerate(loader):
        if k >= n:
            break
        im = batch["img"].to(device).float() / 255.0
        _, p3, p2, p4, p5 = s1(im)
        yA_ref, yB_ref = (o[0] for o in model._predict_once_dual(im))
        dA = ((s2a(p3, p2, p4, p5) - yA_ref).abs().max() / yA_ref.abs().max()).item()
        dB = ((s2b(p3, p4, p5) - yB_ref).abs().max() / yB_ref.abs().max()).item()
        assert dA < tol and dB < tol, f"Stage tidak setara dgn model utuh: {dA:.2e} / {dB:.2e}"
    return True


# =============================================================================
# DATA & VALIDATOR
# =============================================================================
def make_loader(data, split, batch, imgsz=640, workers=2, pin_memory=None):
    """
    Letterbox persegi imgsz, tanpa augmentasi, urutan tetap (sama dengan engine TensorRT fixed-shape).

    SENGAJA DataLoader biasa, BUKAN build_dataloader Ultralytics: InfiniteDataLoader memakai satu iterator
    persisten, sehingga loop yang di-`break` lebih awal (uji kesetaraan, verifikasi ONNX, next(iter(...)))
    menggeser urutan SEMUA iterasi berikutnya (rotasi). Akurasi tetap benar (GT ikut batch), tetapi apa pun
    yang menyejajarkan hasil per gambar dengan dataset.labels / run lain menjadi tidak sejajar.
    """
    from torch.utils.data import DataLoader
    from ultralytics.data import YOLODataset
    ds = YOLODataset(img_path=data[split], data=data, task="detect", imgsz=imgsz,
                     augment=False, rect=False, cache=False, batch_size=batch)
    return DataLoader(ds, batch_size=batch, shuffle=False, num_workers=workers, collate_fn=ds.collate_fn,
                      pin_memory=torch.cuda.is_available() if pin_memory is None else pin_memory, drop_last=False)


def make_validator_cls():
    from ultralytics.models.yolo.detect import DetectionValidator
    from ultralytics.utils.nms import non_max_suppression

    class ListPredValidator(DetectionValidator):
        """Validator resmi Ultralytics yang menerima daftar prediksi per gambar (NMS & threshold identik)."""
        def postprocess(self, preds):
            out = []
            for pred in preds:
                x = non_max_suppression(pred, self.args.conf, self.args.iou, nc=0, multi_label=True,
                                        agnostic=False, max_det=self.args.max_det)[0]
                out.append({"bboxes": x[:, :4], "conf": x[:, 4], "cls": x[:, 5], "extra": x[:, 6:]})
            return out

        def scale_preds(self, predn, pbatch):
            """
            Skala letterbox -> resolusi asli PER SUMBU (hanya untuk JSON/pycocotools; metrik Ultralytics dihitung
            di ruang letterbox dan tidak terpengaruh). ops.scale_boxes bawaan memakai satu gain = rasio TINGGI untuk
            kedua sumbu; pada KITTI (1242x375 -> 640x194, tinggi dibulatkan dari 193.2) rasio tinggi 0.51733 vs
            lebar 0.51530, sehingga koordinat x menyusut 0.39% (±4 px di tepi kanan) -> AP COCO turun, terutama
            objek kecil & IoU tinggi.
            """
            (gh, gw), (px, py) = pbatch["ratio_pad"]
            b = predn["bboxes"].clone()
            b[:, [0, 2]] = (b[:, [0, 2]] - px) / gw
            b[:, [1, 3]] = (b[:, [1, 3]] - py) / gh
            h0, w0 = pbatch["ori_shape"]
            b[:, [0, 2]] = b[:, [0, 2]].clamp(0, w0); b[:, [1, 3]] = b[:, [1, 3]].clamp(0, h0)
            return {**predn, "bboxes": b}
    return ListPredValidator


def run_eval(name, predict_fn, loader, data, cfg, device, save_json=False, verbose=True):
    """predict_fn(img_float_0_1) -> (list[(1,4+nc,N)], list[dict info per gambar])."""
    V = make_validator_cls()
    v = V(args=dict(data=cfg["data_yaml"], conf=cfg["conf"], iou=cfg["iou"], max_det=cfg["max_det"],
                    split=cfg.get("report_split", "test"), imgsz=cfg["imgsz"], batch=1, half=False, plots=False,
                    save_json=save_json, task="detect", single_cls=False, agnostic_nms=False))
    v.data, v.device = data, device
    v.init_metrics(SimpleNamespace(names=data["names"], end2end=False))
    records = []
    with torch.no_grad():
        for batch in loader:
            batch = v.preprocess(batch)
            preds, infos = predict_fn(batch["img"])
            v.update_metrics(v.postprocess(preds), batch)
            for f, info in zip(batch["im_file"], infos):
                records.append({"file": Path(f).name, **info})
    stats = v.get_stats()
    if verbose:
        print(f"--- {name} ---"); v.print_results()
    assert v.seen == len(loader.dataset), f"{name}: {v.seen} gambar dievaluasi, dataset {len(loader.dataset)}"
    return dict(name=name, stats=stats, records=records, jdict=list(v.jdict) if save_json else None, seen=v.seen)


def summarize_acc(r):
    s = r["stats"]
    return {"mAP50": s["metrics/mAP50(B)"], "mAP50-95": s["metrics/mAP50-95(B)"],
            "P": s["metrics/precision(B)"], "R": s["metrics/recall(B)"], "n_images": r["seen"]}


# =============================================================================
# CACHE PER GAMBAR (satu forward; semua mode evaluasi dirakit dari sini)
# =============================================================================
@torch.no_grad()
def cache_split(model, stages, loader, device, keep_outputs=True):
    """
    Per gambar: z (logit gate mentah), output Stage2A & Stage2B (CPU), loss per-sample A/B (total & small),
    jumlah objek kecil ter-assign, info GT (resolusi asli, definisi COCO). Loader harus batch=1, shuffle=False.
    """
    s1, s2a, s2b = stages
    rt = find_router(model)
    crit = model.init_criterion()
    labels = loader.dataset.labels
    out = dict(file=[], z=[], yA=[], yB=[], la=[], lb=[], las=[], lbs=[], nsa=[], nsb=[],
               n_obj=[], n_small=[], small_ratio=[], mean_area=[])
    for i, batch in enumerate(loader):
        assert batch["img"].shape[0] == 1, "cache_split butuh loader batch=1"
        f = batch["im_file"][0]
        assert Path(f).name == Path(labels[i]["im_file"]).name, "Urutan loader != dataset.labels"
        im = batch["img"].to(device).float() / 255.0
        _, p3, p2, p4, p5 = s1(im)
        out["z"].append(float(_raw_gate_logit(rt, p3, p2)[0]))
        if keep_outputs:
            out["yA"].append(s2a(p3, p2, p4, p5).float().cpu())
            out["yB"].append(s2b(p3, p4, p5).float().cpu())
        b = {"img": im, **{k: batch[k].to(device) for k in ("cls", "bboxes", "batch_idx")}}
        det_A, det_B = model._predict_once_dual(im)
        crit.loss_A(det_A, b); crit.loss_B(det_B, b)
        out["la"].append(float(crit.loss_A._last_per_sample_loss[0]))
        out["lb"].append(float(crit.loss_B._last_per_sample_loss[0]))
        out["las"].append(float(crit.loss_A._last_per_sample_loss_small[0]))
        out["lbs"].append(float(crit.loss_B._last_per_sample_loss_small[0]))
        out["nsa"].append(float(crit.loss_A._last_small_obj_count[0]))
        out["nsb"].append(float(crit.loss_B._last_small_obj_count[0]))
        lab = labels[i]
        h, w = lab["shape"]
        bx = lab["bboxes"].reshape(-1, 4)
        area = bx[:, 2] * w * bx[:, 3] * h
        out["file"].append(Path(f).name)
        out["n_obj"].append(int(len(bx)))
        out["n_small"].append(int((area < 32 ** 2).sum()))
        out["small_ratio"].append(float((area < 32 ** 2).mean()) if len(bx) else np.nan)
        out["mean_area"].append(float(area.mean()) if len(bx) else np.nan)
    for k in ("z", "la", "lb", "las", "lbs", "nsa", "nsb", "n_obj", "n_small", "small_ratio", "mean_area"):
        out[k] = np.asarray(out[k], dtype=np.float64)
    out["oracle_A"] = out["la"] < out["lb"]                       # 1 = Branch A lebih baik (sinyal training 'total')
    out["valid_small"] = (out["nsa"] > 0) & (out["nsb"] > 0)
    out["oracle_A_small"] = out["las"] < out["lbs"]
    return out


def predict_from_cache(cache, use_a, device):
    """predict_fn untuk run_eval dari cache; loader harus urutan sama (dicek lewat records)."""
    pos = {"k": 0}

    def fn(im):
        preds, infos = [], []
        for _ in range(im.shape[0]):
            k = pos["k"]; pos["k"] += 1
            a = bool(use_a[k])
            # clone WAJIB: non_max_suppression Ultralytics menulis xywh->xyxy IN-PLACE pada tensor input;
            # tanpa salinan, cache rusak untuk evaluasi berikutnya (di CPU .to() tidak menyalin).
            preds.append((cache["yA"][k] if a else cache["yB"][k]).to(device, copy=True))
            infos.append({"branch": "A" if a else "B", "z": float(cache["z"][k])})
        return preds, infos
    return fn


def eval_assignment(name, cache, use_a, loader, data, cfg, device, save_json=False, verbose=False):
    r = run_eval(name, predict_from_cache(cache, use_a, device), loader, data, cfg, device,
                 save_json=save_json, verbose=verbose)
    assert [x["file"] for x in r["records"]] == cache["file"], "Urutan loader != cache"
    r["activation"] = float(np.mean(use_a))
    r["use_a"] = np.asarray(use_a, bool)
    return r


def random_assignment(n, k, seed):
    rng = np.random.default_rng(seed)
    ua = np.zeros(n, bool)
    ua[rng.choice(n, k, replace=False)] = True
    return ua


# =============================================================================
# TITIK OPERASI & KALIBRASI PERSENTIL
# =============================================================================
def fit_percentile_threshold(raw_logits, target_activation):
    """T = 1, b = -q dgn q = kuantil (1 - rho) logit val -> aktivasi val = rho (selain ties)."""
    assert 0.0 < target_activation < 1.0
    z = torch.as_tensor(raw_logits, dtype=torch.float64)
    q = torch.quantile(z, 1.0 - target_activation).item()
    return dict(calib_T=1.0, calib_b=-q, threshold=q, target_activation=float(target_activation),
                activation_on_calib_split=float((z > q).double().mean()),
                n_unique_logits=int(z.unique().numel()), n=int(z.numel()))


def sweep(cache, loader, data, cfg, device, thr_source, rates, n_random=3, seed=0, verbose=False):
    """
    Router (threshold = kuantil logit thr_source) vs acak (aktivasi sama, n_random undian) per rate.
    thr_source = logit VAL, juga saat mengevaluasi test (threshold tidak pernah dari test).
    """
    z = cache["z"]; n = len(z); rows = []
    for r in rates:
        thr = -np.inf if r >= 1 else (np.inf if r <= 0 else float(np.quantile(thr_source, 1 - r)))
        use_a = z > thr
        k = int(use_a.sum())
        acc = summarize_acc(eval_assignment(f"router r={r:.2f}", cache, use_a, loader, data, cfg, device, verbose=verbose))
        rnd = []
        for j in range(n_random if 0 < k < n else 0):
            ua = random_assignment(n, k, seed + 1000 * j + int(r * 100))
            rnd.append(summarize_acc(eval_assignment(f"acak r={k / n:.2f}", cache, ua, loader, data, cfg, device))["mAP50"])
        rows.append(dict(rate_target=r, rate_realized=k / n, threshold=thr if np.isfinite(thr) else None,
                         mAP50=acc["mAP50"], mAP50_95=acc["mAP50-95"],
                         acak_mAP50=float(np.mean(rnd)) if rnd else acc["mAP50"],
                         acak_sd=float(np.std(rnd)) if rnd else 0.0, n_acak=len(rnd)))
    df = pd.DataFrame(rows)
    df["gain_vs_acak"] = df.mAP50 - df.acak_mAP50
    return df


def choose_r_star(df_val):
    """Aturan terkunci: r* = argmax (mAP50 router - mAP50 acak) pada 0 < rho < 1, di VAL."""
    inner = df_val[(df_val.rate_target > 0) & (df_val.rate_target < 1)]
    return float(inner.loc[inner.gain_vs_acak.idxmax(), "rate_target"])


# =============================================================================
# COCO + BOOTSTRAP BERPASANGAN PER GAMBAR
# =============================================================================
def build_coco_gt(dataset, names, path):
    """GT COCO resolusi asli; category_id = cls+1, image_id = int(stem) (= pred_to_json Ultralytics)."""
    from pycocotools.coco import COCO
    images, anns, aid = [], [], 1
    for lab in dataset.labels:
        stem = Path(lab["im_file"]).stem
        iid = int(stem) if stem.isnumeric() else stem
        h, w = lab["shape"]
        images.append({"id": iid, "file_name": Path(lab["im_file"]).name, "height": int(h), "width": int(w)})
        for c, (cx, cy, bw, bh) in zip(lab["cls"].reshape(-1), lab["bboxes"].reshape(-1, 4)):
            bw_px, bh_px = float(bw * w), float(bh * h)
            anns.append({"id": aid, "image_id": iid, "category_id": int(c) + 1,
                         "bbox": [float(cx * w - bw_px / 2), float(cy * h - bh_px / 2), bw_px, bh_px],
                         "area": bw_px * bh_px, "iscrowd": 0})
            aid += 1
    gt = {"images": images, "annotations": anns, "categories": [{"id": i + 1, "name": str(n)} for i, n in names.items()]}
    Path(path).write_text(json.dumps(gt))
    return COCO(str(path))


COCO_KEYS = ["mAP50-95", "mAP50", "mAP75", "mAP_small", "mAP_medium", "mAP_large",
             "AR1", "AR10", "AR100", "AR_small", "AR_medium", "AR_large"]


def coco_eval(coco_gt, jdict, quiet=True):
    """COCOeval lengkap. Returns (stats dict, objek COCOeval — evalImgs dipakai bootstrap)."""
    import contextlib, io
    from pycocotools.cocoeval import COCOeval
    ids = set(coco_gt.getImgIds())
    assert {d["image_id"] for d in jdict} <= ids, "image_id prediksi tidak ada di GT"
    buf = io.StringIO()
    with (contextlib.redirect_stdout(buf) if quiet else contextlib.nullcontext()):
        if len(jdict):
            dt = coco_gt.loadRes(jdict)
        else:                                               # tanpa deteksi sama sekali -> semua AP = 0
            from pycocotools.coco import COCO
            dt = COCO(); dt.dataset = {"images": coco_gt.dataset["images"], "categories": coco_gt.dataset["categories"],
                                      "annotations": []}
            dt.createIndex()
        E = COCOeval(coco_gt, dt, "bbox")
        E.params.imgIds = sorted(ids)
        E.evaluate(); E.accumulate(); E.summarize()
    return dict(zip(COCO_KEYS, map(float, E.stats))), E


class CocoBootstrap:
    """
    AP COCO untuk subset gambar (boleh berulang, untuk bootstrap) langsung dari evalImgs COCOeval:
    replikasi accumulate() + summarize() (maxDet 100) tanpa menjalankan evaluate() ulang.
    Diverifikasi sama dengan COCOeval.stats pada himpunan penuh (lihat verify()).
    """
    AREAS = {"all": 0, "small": 1, "medium": 2, "large": 3}

    def __init__(self, E):
        p = E.params
        self.T, self.R, self.I = len(p.iouThrs), p.recThrs, len(p.imgIds)
        self.A, self.K, self.maxDet = len(p.areaRng), len(p.catIds), p.maxDets[-1]
        self.iou50 = int(np.where(np.isclose(p.iouThrs, 0.5))[0][0])
        self.stats_full = np.asarray(E.stats, float)
        # pra-ekstrak array per (kategori, area, gambar); None = tidak ada GT & deteksi
        self.cells = {}
        for k in range(self.K):
            for name, a in self.AREAS.items():
                base = k * self.A * self.I + a * self.I
                lst = []
                for i in range(self.I):
                    e = E.evalImgs[base + i]
                    if e is None:
                        lst.append(None); continue
                    lst.append((np.asarray(e["dtScores"][:self.maxDet], float),
                                np.asarray(e["dtMatches"][:, :self.maxDet]) > 0,
                                np.asarray(e["dtIgnore"][:, :self.maxDet], bool),
                                int(np.count_nonzero(np.asarray(e["gtIgnore"]) == 0))))
                self.cells[(k, name)] = lst

    def _precision(self, idx, area):
        """List per kategori: array (T, R) presisi terinterpolasi, atau None (kategori tanpa GT valid)."""
        out = []
        for k in range(self.K):
            cell = self.cells[(k, area)]
            E = [cell[i] for i in idx if cell[i] is not None]
            if not E:
                out.append(None); continue
            npig = sum(e[3] for e in E)
            if npig == 0:
                out.append(None); continue
            scores = np.concatenate([e[0] for e in E])
            order = np.argsort(-scores, kind="mergesort")
            dtm = np.concatenate([e[1] for e in E], axis=1)[:, order]
            dtig = np.concatenate([e[2] for e in E], axis=1)[:, order]
            tp = np.cumsum(dtm & ~dtig, axis=1, dtype=float)
            fp = np.cumsum(~dtm & ~dtig, axis=1, dtype=float)
            nd = tp.shape[1]
            q = np.zeros((self.T, len(self.R)))
            if nd:
                rc = tp / npig
                pr = tp / (fp + tp + np.spacing(1))
                pr = np.maximum.accumulate(pr[:, ::-1], axis=1)[:, ::-1]      # presisi monoton (= loop COCO)
                for t in range(self.T):
                    ii = np.searchsorted(rc[t], self.R, side="left")
                    ok = ii < nd
                    q[t, ok] = pr[t, ii[ok]]
            out.append(q)
        return out

    def metrics(self, idx):
        """dict mAP50-95, mAP50 (area all) & mAP_small untuk subset gambar idx (indeks ke E.params.imgIds)."""
        P_all = [p for p in self._precision(idx, "all") if p is not None]
        P_s = [p for p in self._precision(idx, "small") if p is not None]
        m = {}
        if P_all:
            P = np.stack(P_all); m["mAP50-95"] = float(P.mean()); m["mAP50"] = float(P[:, self.iou50].mean())
        else:
            m["mAP50-95"] = m["mAP50"] = np.nan
        m["mAP_small"] = float(np.stack(P_s).mean()) if P_s else np.nan
        return m

    def verify(self, atol=1e-9):
        full = self.metrics(np.arange(self.I))
        ref = {"mAP50-95": self.stats_full[0], "mAP50": self.stats_full[1], "mAP_small": self.stats_full[3]}
        for k, v in ref.items():
            v = np.nan if v < 0 else v
            assert (np.isnan(v) and np.isnan(full[k])) or abs(full[k] - v) <= atol, f"CocoBootstrap {k}: {full[k]} != COCOeval {v}"
        return full


BOOT_METRICS = ("mAP50-95", "mAP50", "mAP_small")


def paired_bootstrap(boot_a, boots_b, n_boot=1000, seed=0):
    """
    Selisih metrik A − rata-rata(B_list) pada resample gambar YANG SAMA (bootstrap berpasangan per gambar).
    boot_a: CocoBootstrap; boots_b: list CocoBootstrap (mis. 3 undian router acak) — semua dari GT yang sama.
    Returns DataFrame: metric, diff, ci_lo, ci_hi (persentil 95%), p_two_sided (proporsi bootstrap melewati 0;
    batas bawah 1/B — tulis 'p < 1/B', bukan p = 0).
    """
    I = boot_a.I
    assert all(b.I == I for b in boots_b)
    rng = np.random.default_rng(seed)

    def diff(idx):
        a = boot_a.metrics(idx)
        bs = [b.metrics(idx) for b in boots_b]
        return {m: a[m] - np.nanmean([b[m] for b in bs]) for m in BOOT_METRICS}

    point = diff(np.arange(I))
    samples = {m: [] for m in BOOT_METRICS}
    for _ in range(n_boot):
        d = diff(rng.integers(0, I, I))
        for m in BOOT_METRICS:
            samples[m].append(d[m])
    rows = []
    for m in BOOT_METRICS:
        s = np.asarray(samples[m]); s = s[np.isfinite(s)]
        lo, hi = np.percentile(s, [2.5, 97.5])
        p = min(1.0, max(2 * min((s <= 0).mean(), (s >= 0).mean()), 1.0 / len(s)))   # resolusi p = 1/B
        rows.append(dict(metric=m, diff=point[m], ci_lo=lo, ci_hi=hi, p_two_sided=p, n_boot=len(s)))
    return pd.DataFrame(rows)


# =============================================================================
# PERILAKU GATE (RQ3)
# =============================================================================
def holm(pvals):
    p = np.asarray(pvals, float); m = len(p); order = np.argsort(p)
    adj = np.empty(m); run = 0.0
    for r, i in enumerate(order):
        run = max(run, min(1.0, (m - r) * p[i])); adj[i] = run
    return adj


def gate_behaviour(cache):
    """Spearman z vs kepadatan/ukuran, parsial (proporsi kecil | jumlah objek), AUC vs oracle; Holm."""
    from scipy import stats as st
    df = pd.DataFrame({k: cache[k] for k in ("z", "n_obj", "n_small", "small_ratio", "mean_area")})
    tests = []

    def sp(x, y, label):
        m = df[[x, y]].dropna()
        r, p = st.spearmanr(m[x], m[y]); tests.append(dict(test=label, r=r, p=p, n=len(m)))

    sp("z", "n_obj", "Spearman z vs jumlah objek")
    sp("z", "small_ratio", "Spearman z vs proporsi objek kecil")
    sp("z", "n_small", "Spearman z vs jumlah objek kecil")
    sp("z", "mean_area", "Spearman z vs rata-rata luas objek")
    m = df[["z", "small_ratio", "n_obj"]].dropna()
    rx, ry, rz = st.rankdata(m.z), st.rankdata(m.small_ratio), st.rankdata(m.n_obj)
    res = lambda a, b: a - np.polyval(np.polyfit(b, a, 1), b)
    r_part, p_part = st.pearsonr(res(rx, rz), res(ry, rz))
    tests.append(dict(test="Parsial (Spearman) z vs proporsi kecil | jumlah objek (H3)", r=r_part, p=p_part, n=len(m)))
    T = pd.DataFrame(tests); T["p_holm"] = holm(T.p)
    auc = _auc_with_ci(cache["oracle_A"], cache["z"])
    v = cache["valid_small"]
    auc_s = _auc_with_ci(cache["oracle_A_small"][v], cache["z"][v]) if v.sum() >= 20 else (np.nan, np.nan, np.nan)
    A = pd.DataFrame([dict(target="loss_A < loss_B (total)", n=len(cache["z"]), oracle_rate=float(cache["oracle_A"].mean()),
                           auc=auc[0], ci_lo=auc[1], ci_hi=auc[2]),
                      dict(target="loss_A_small < loss_B_small (gambar valid)", n=int(v.sum()),
                           oracle_rate=float(cache["oracle_A_small"][v].mean()) if v.sum() else np.nan,
                           auc=auc_s[0], ci_lo=auc_s[1], ci_hi=auc_s[2])])
    return T, A


# =============================================================================
# DIAGNOSTIK BATCHNORM GATE (hipotesis close_mosaic)
# =============================================================================
def _squeeze_bns(rt):
    return [m for seq in (rt.squeeze_p3, rt.squeeze_p2) for m in seq.modules() if isinstance(m, nn.BatchNorm2d)]


@torch.no_grad()
def gate_logits_on(model, loader, device, max_images=None):
    """z per gambar lewat forward penuh model (hook input router) — mode BN mengikuti status modul."""
    rt = find_router(model)
    feats, zs, files = {}, [], []
    h = rt.register_forward_pre_hook(lambda m, inp: feats.update(p3=inp[0][0], p2=inp[0][1]))
    try:
        n = 0
        for batch in loader:
            im = batch["img"].to(device).float() / 255.0
            model._predict_once_dual(im)
            zs.append(_raw_gate_logit(rt, feats["p3"], feats["p2"]).cpu().numpy())
            files += [Path(f).name for f in batch["im_file"]]
            n += im.shape[0]
            if max_images and n >= max_images:
                break
    finally:
        h.remove()
    return np.concatenate(zs), files


@torch.no_grad()
def bn_diagnostic(model, val_cache, val_loader_b16, train_loader_b16, device, n_train=1297):
    """
    Bandingkan peringkat gate (AUC vs oracle 'total' val) & sebaran logit di val untuk:
      (a) eval (running stats)             — referensi, harus = cache
      (b) BN squeeze memakai statistik batch val (batch 16) — meniru mode training
      (c) running stats BN squeeze diestimasi ulang di gambar TRAIN tanpa augmentasi (kumulatif)
      (d) gambar TRAIN tanpa augmentasi, mode eval — apakah gate berbeda di data training?
    Semua pada salinan model; model asli tidak berubah.
    """
    rows = []
    oracle = dict(zip(val_cache["file"], val_cache["oracle_A"]))

    def row(name, z, files, labels=None):
        y = np.array([oracle[f] for f in files]) if labels is None else labels
        a = _auc_with_ci(y, z, n_boot=500)
        rows.append(dict(variant=name, n=len(z), logit_mean=float(np.mean(z)), logit_std=float(np.std(z)),
                         act_z_gt_0=float(np.mean(z > 0)), auc=a[0], ci_lo=a[1], ci_hi=a[2]))

    z, f = gate_logits_on(model, val_loader_b16, device); row("(a) val, eval (running stats)", z, f)
    mb = copy.deepcopy(model)
    for bn in _squeeze_bns(find_router(mb)):
        bn.train()
    z, f = gate_logits_on(mb, val_loader_b16, device); row("(b) val, BN squeeze statistik batch", z, f)
    mc = copy.deepcopy(model); rt = find_router(mc)
    for bn in _squeeze_bns(rt):
        bn.reset_running_stats(); bn.momentum = None; bn.train()
    gate_logits_on(mc, train_loader_b16, device, max_images=n_train)       # hanya mengisi running stats
    for bn in _squeeze_bns(rt):
        bn.eval()
    z, f = gate_logits_on(mc, val_loader_b16, device); row("(c) val, running stats BN squeeze diestimasi ulang di train", z, f)
    return pd.DataFrame(rows)


def train_split_gate(model, train_loader_b1, device, max_images=1297):
    """(d) Gate & oracle di gambar train tanpa augmentasi, mode eval (dipotong max_images)."""
    stages = make_stages(model)
    sub = _LimitedLoader(train_loader_b1, max_images)
    c = cache_split(model, stages, sub, device, keep_outputs=False)
    a = _auc_with_ci(c["oracle_A"], c["z"], n_boot=500)
    return dict(variant="(d) train tanpa augmentasi, eval", n=len(c["z"]), logit_mean=float(c["z"].mean()),
                logit_std=float(c["z"].std()), act_z_gt_0=float((c["z"] > 0).mean()), auc=a[0], ci_lo=a[1], ci_hi=a[2])


class _LimitedLoader:
    """Bungkus loader: berhenti setelah n gambar; dataset.labels dipotong agar cache_split tetap sinkron."""
    def __init__(self, loader, n):
        self.loader, self.n = loader, n
        self.dataset = SimpleNamespace(labels=loader.dataset.labels[:n])

    def __iter__(self):
        for i, b in enumerate(self.loader):
            if i >= self.n:
                break
            yield b


# =============================================================================
# BASELINE STATIS (Vanilla, P2-Static)
# =============================================================================
def load_static(path, device):
    from ultralytics import YOLO
    m = YOLO(path).model.float().to(device).eval()
    for p in m.parameters():
        p.requires_grad_(False)
    return m


def predict_static_pt(m):
    @torch.no_grad()
    def fn(im):
        preds = []
        for i in range(im.shape[0]):
            out = m(im[i:i + 1])
            preds.append(out[0] if isinstance(out, (tuple, list)) else out)
        return preds, [{} for _ in range(im.shape[0])]
    return fn


# =============================================================================
# ONNX & TENSORRT (FP32, 3 engine router + 2 baseline; sama dgn notebook uji 2026-10-08)
# =============================================================================
class _Stage1Out(nn.Module):
    def __init__(self, s):
        super().__init__(); self.s = s

    def forward(self, x):
        return self.s(x)


class _StaticOut(nn.Module):
    def __init__(self, m):
        super().__init__(); self.m = m

    def forward(self, x):
        out = self.m(x)
        return out[0] if isinstance(out, (tuple, list)) else out


def export_onnx(module, args, path, in_names, out_names):
    with torch.no_grad():
        torch.onnx.export(module.eval(), args, str(path), input_names=in_names, output_names=out_names,
                          opset_version=17, dynamic_axes=None, dynamo=False)


def cpu_copy(m):
    """
    Salinan model di CPU yang aman untuk ekspor. Detect menyimpan anchors/strides sebagai ATRIBUT biasa (bukan
    buffer) dan hanya menghitung ulang bila shape input berubah; .cpu() tidak memindahkannya, sehingga salinan
    CPU dari model yang pernah jalan di GPU masih memegang anchors cuda:0 ("Expected all tensors to be on the
    same device"). shape=None memaksa anchors dihitung ulang di device yang benar pada forward berikutnya.
    """
    c = copy.deepcopy(m).cpu().eval()
    for mod in c.modules():
        if hasattr(mod, "anchors") and hasattr(mod, "shape"):
            mod.shape = None
            mod.anchors, mod.strides = torch.empty(0), torch.empty(0)
    return c


def export_all_onnx(model, vanilla, p2static, x0, out_dir, tag):
    """x0: (1,3,H,W) float CPU. Returns dict path ONNX + salinan CPU stage (untuk verifikasi)."""
    m_cpu = cpu_copy(model)
    assert get_calibration(find_router(m_cpu)) == get_calibration(find_router(model)), "Kalibrasi tidak ikut tersalin"
    s1c, s2ac, s2bc = make_stages(m_cpu)
    with torch.no_grad():
        _, p3, p2, p4, p5 = s1c(x0)
    P = {k: Path(out_dir) / f"{k}_{tag}.onnx" for k in ("stage1", "stage2a", "stage2b")}
    P["vanilla"], P["p2static"] = Path(out_dir) / "vanilla.onnx", Path(out_dir) / "p2static.onnx"
    export_onnx(_Stage1Out(s1c), (x0,), P["stage1"], ["image"], ["gate", "p3_neck", "p2_backbone", "p4_neck", "p5_sppf"])
    export_onnx(s2ac, (p3, p2, p4, p5), P["stage2a"], ["p3_neck", "p2_backbone", "p4_neck", "p5_sppf"], ["y_A"])
    export_onnx(s2bc, (p3, p4, p5), P["stage2b"], ["p3_neck", "p4_neck", "p5_sppf"], ["y_B"])
    statics = {"vanilla": _StaticOut(cpu_copy(vanilla)).eval(), "p2static": _StaticOut(cpu_copy(p2static)).eval()}
    for k, mod in statics.items():
        export_onnx(mod, (x0,), P[k], ["image"], ["y"])
    return P, (s1c, s2ac, s2bc), statics


def verify_onnx(P, cpu_stages, statics, loader, n=5):
    import onnxruntime as ort
    sess = {k: ort.InferenceSession(str(p), providers=["CPUExecutionProvider"]) for k, p in P.items()}
    run = lambda k, feed: sess[k].run(None, {i.name: feed[i.name] for i in sess[k].get_inputs()})
    rel = lambda a, b: float(np.abs(a - b).max() / (np.abs(b).max() + 1e-12))
    s1c, s2ac, s2bc = cpu_stages
    worst = {}
    with torch.no_grad():
        for k, batch in enumerate(loader):
            if k >= n:
                break
            x = batch["img"].float() / 255.0
            g, p3, p2, p4, p5 = s1c(x)
            o1 = run("stage1", {"image": x.numpy()})
            assert float(o1[0].flatten()[0]) == float(g.flatten()[0]), f"Gate ONNX != PyTorch pada {batch['im_file'][0]}"
            worst["stage1"] = max(worst.get("stage1", 0), *[rel(a, b.numpy()) for a, b in zip(o1[1:], (p3, p2, p4, p5))])
            f = {"p3_neck": p3.numpy(), "p2_backbone": p2.numpy(), "p4_neck": p4.numpy(), "p5_sppf": p5.numpy()}
            worst["stage2a"] = max(worst.get("stage2a", 0), rel(run("stage2a", f)[0], s2ac(p3, p2, p4, p5).numpy()))
            worst["stage2b"] = max(worst.get("stage2b", 0), rel(run("stage2b", f)[0], s2bc(p3, p4, p5).numpy()))
            for nm, mod in statics.items():
                worst[nm] = max(worst.get(nm, 0), rel(run(nm, {"image": x.numpy()})[0], mod(x).numpy()))
    assert all(v < 1e-4 for v in worst.values()), f"Selisih ONNX vs PyTorch > 1e-4: {worst}"
    return worst


def init_trt():
    """Import TensorRT + pycuda (buat context CUDA). Returns namespace atau None bila tidak tersedia."""
    try:
        import tensorrt as trt
        import pycuda.driver as cuda
        import pycuda.autoinit  # noqa: F401
    except Exception as e:
        print("TensorRT/pycuda tidak tersedia:", e)
        return None
    try:
        cuda.Device(0)                                  # GPU dicek lewat pycuda: torch.cuda TIDAK disentuh
    except Exception as e:
        print("GPU tidak tersedia untuk pycuda:", e)
        return None
    NS = SimpleNamespace(trt=trt, cuda=cuda, logger=trt.Logger(trt.Logger.WARNING))

    def build_engine(onnx_path, engine_path):
        builder = trt.Builder(NS.logger)
        network = builder.create_network()
        parser = trt.OnnxParser(network, NS.logger)
        with open(onnx_path, "rb") as f:
            if not parser.parse(f.read()):
                raise RuntimeError("\n".join(str(parser.get_error(i)) for i in range(parser.num_errors)))
        config = builder.create_builder_config()
        config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 1 << 30)   # tanpa flag presisi = FP32
        eb = builder.build_serialized_network(network, config)
        assert eb is not None, f"Gagal build {onnx_path}"
        Path(engine_path).write_bytes(eb)

    def _io(engine):
        inp, out = [], []
        for i in range(engine.num_io_tensors):
            n = engine.get_tensor_name(i)
            (inp if engine.get_tensor_mode(n) == trt.TensorIOMode.INPUT else out).append(n)
        return inp, out

    class TRTEngineOptimized:
        """Buffer dialokasikan sekali. Vanilla & P2-Static."""
        def __init__(self, path):
            with open(path, "rb") as f, trt.Runtime(NS.logger) as rt:
                self.engine = rt.deserialize_cuda_engine(f.read())
            self.ctx = self.engine.create_execution_context(); self.stream = cuda.Stream()
            self.inp, self.out = _io(self.engine); self.d, self.h = {}, {}
            for n in self.inp:
                shape, dt = tuple(self.engine.get_tensor_shape(n)), trt.nptype(self.engine.get_tensor_dtype(n))
                self.ctx.set_input_shape(n, shape)
                self.d[n] = cuda.mem_alloc(int(np.prod(shape)) * np.dtype(dt).itemsize)
                self.ctx.set_tensor_address(n, int(self.d[n]))
            for n in self.out:
                self.h[n] = cuda.pagelocked_empty(tuple(self.ctx.get_tensor_shape(n)), trt.nptype(self.engine.get_tensor_dtype(n)))
                self.d[n] = cuda.mem_alloc(self.h[n].nbytes)
                self.ctx.set_tensor_address(n, int(self.d[n]))

        def infer(self, image_np):
            cuda.memcpy_htod_async(self.d[self.inp[0]], image_np, self.stream)
            self.ctx.execute_async_v3(stream_handle=self.stream.handle)
            for n in self.out:
                cuda.memcpy_dtoh_async(self.h[n], self.d[n], self.stream)
            self.stream.synchronize()
            return self.h[self.out[0]]

    class TRTEngineChainable:
        """Router: Stage1 dari host; Stage2 menerima pointer device Stage1 (tanpa salinan fitur)."""
        def __init__(self, path):
            with open(path, "rb") as f, trt.Runtime(NS.logger) as rt:
                self.engine = rt.deserialize_cuda_engine(f.read())
            self.ctx = self.engine.create_execution_context(); self.stream = cuda.Stream()
            self.inp, self.out = _io(self.engine); self.d_in, self.d_out, self.h_out = {}, {}, {}
            for n in self.inp:
                shape, dt = tuple(self.engine.get_tensor_shape(n)), trt.nptype(self.engine.get_tensor_dtype(n))
                self.ctx.set_input_shape(n, shape)
                self.d_in[n] = cuda.mem_alloc(int(np.prod(shape)) * np.dtype(dt).itemsize)
            for n in self.out:
                self.h_out[n] = cuda.pagelocked_empty(tuple(self.ctx.get_tensor_shape(n)), trt.nptype(self.engine.get_tensor_dtype(n)))
                self.d_out[n] = cuda.mem_alloc(self.h_out[n].nbytes)
                self.ctx.set_tensor_address(n, int(self.d_out[n]))

        def run_from_host(self, image_np):
            n = self.inp[0]
            cuda.memcpy_htod_async(self.d_in[n], image_np, self.stream)
            self.ctx.set_tensor_address(n, int(self.d_in[n]))
            self.ctx.execute_async_v3(stream_handle=self.stream.handle)

        def read_scalar(self, name):
            cuda.memcpy_dtoh_async(self.h_out[name], self.d_out[name], self.stream)
            self.stream.synchronize()
            return float(self.h_out[name].flatten()[0])

        def run_from_device(self, ptrs):
            for n, p in ptrs.items():
                self.ctx.set_tensor_address(n, int(p))
            self.ctx.execute_async_v3(stream_handle=self.stream.handle)
            for n in self.out:
                cuda.memcpy_dtoh_async(self.h_out[n], self.d_out[n], self.stream)
            self.stream.synchronize()
            return self.h_out[self.out[0]]

    NS.build_engine, NS.Optimized, NS.Chainable = build_engine, TRTEngineOptimized, TRTEngineChainable
    return NS


class TRTSystem:
    """Router 3-engine + 2 baseline. router(image_np, force=None|'A'|'B') -> (y, gate, branch)."""
    A_KEYS, B_KEYS = ("p3_neck", "p2_backbone", "p4_neck", "p5_sppf"), ("p3_neck", "p4_neck", "p5_sppf")

    def __init__(self, NS, engines):
        self.s1, self.s2a, self.s2b = (NS.Chainable(engines[k]) for k in ("stage1", "stage2a", "stage2b"))
        self.vanilla, self.p2static = NS.Optimized(engines["vanilla"]), NS.Optimized(engines["p2static"])

    def router(self, image_np, force=None):
        self.s1.run_from_host(image_np)
        g = self.s1.read_scalar("gate")
        use_a = (g > 0.5) if force is None else (force == "A")
        if use_a:
            y = self.s2a.run_from_device({k: self.s1.d_out[k] for k in self.A_KEYS})
        else:
            y = self.s2b.run_from_device({k: self.s1.d_out[k] for k in self.B_KEYS})
        return y, g, "A" if use_a else "B"


def predict_router_trt(sysm):
    def fn(im):
        preds, infos = [], []
        for i in range(im.shape[0]):
            y, g, br = sysm.router(np.ascontiguousarray(im[i:i + 1].cpu().numpy(), dtype=np.float32))
            preds.append(torch.from_numpy(y.copy()).to(im.device)); infos.append({"gate": g, "branch": br})
        return preds, infos
    return fn


def predict_static_trt(eng):
    def fn(im):
        preds = [torch.from_numpy(eng.infer(np.ascontiguousarray(im[i:i + 1].cpu().numpy(), dtype=np.float32)).copy()).to(im.device)
                 for i in range(im.shape[0])]
        return preds, [{} for _ in range(im.shape[0])]
    return fn


def lat_stats(a):
    a = np.asarray(a, float)
    return dict(mean=a.mean(), std=a.std(), p50=np.percentile(a, 50), p95=np.percentile(a, 95),
                p99=np.percentile(a, 99), min=a.min(), max=a.max(), n=int(a.size))


def benchmark_latency(sysm, loader, n_warmup=50, n_reps=3):
    """
    Interleaved (urutan dibalik tiap putaran), gambar test asli yang sama, cakupan: H2D gambar + inferensi
    (+ baca 1 skalar gate) + D2H output. Router dipaksa A / B diukur untuk kurva latensi t(r) = r·T_A + (1−r)·T_B.
    """
    as_np = lambda b: np.ascontiguousarray(b["img"].numpy().astype(np.float32) / 255.0)
    for k, batch in enumerate(loader):
        if k >= n_warmup:
            break
        im = as_np(batch)
        sysm.vanilla.infer(im); sysm.p2static.infer(im); sysm.router(im, "A"); sysm.router(im, "B")
    fns = {"Vanilla": lambda im: sysm.vanilla.infer(im), "P2-Static": lambda im: sysm.p2static.infer(im),
           "Router": lambda im: sysm.router(im), "Router paksa A": lambda im: sysm.router(im, "A"),
           "Router paksa B": lambda im: sysm.router(im, "B")}
    t = {k: [] for k in fns}
    branches, by_branch = [], {"A": [], "B": []}
    for rep in range(n_reps):
        for batch in loader:
            im = as_np(batch)
            order = list(fns) if rep % 2 == 0 else list(fns)[::-1]
            for name in order:
                t0 = time.perf_counter(); out = fns[name](im); dt = (time.perf_counter() - t0) * 1e3
                t[name].append(dt)
                if name == "Router":
                    by_branch[out[2]].append(dt)
                    if rep == 0:
                        branches.append(out[2])
    LAT = {k: lat_stats(v) for k, v in t.items()}
    for b in "AB":
        LAT[f"Router (saat Branch {b})"] = lat_stats(by_branch[b]) if by_branch[b] else None
    return LAT, float(np.mean([b == "A" for b in branches]))


# =============================================================================
# JOB TENSORRT DI PROSES TERPISAH  (python -m tahap2_eval <trt_job.json>)
# =============================================================================
def _rss_gb():
    try:
        import psutil
        return psutil.Process().memory_info().rss / 1e9
    except Exception:
        return float("nan")


def trt_job_main(job_path):
    """
    Build engine + mAP dari output TensorRT + latensi, di proses TERPISAH dari kernel notebook:
      - kernel notebook memegang cache besar (output Branch A/B, jdict, COCOeval) -> RAM kernel tidak ikut
        terbebani builder TensorRT;
      - proses ini TIDAK pernah membuat context CUDA PyTorch (validator & NMS di CPU), sehingga hanya ada satu
        context (pycuda) -> tidak ada konflik dua context di satu proses;
      - bila TensorRT/pycuda crash (segfault) atau proses dibunuh OOM, kernel notebook tetap hidup dan
        returncode memberi tahu penyebabnya.
    Masukan: JSON (onnx, data_yaml, split, imgsz, conf, iou, max_det, gt_json, out_json, n_warmup, n_reps).
    """
    job = json.loads(Path(job_path).read_text())
    t0 = time.perf_counter()
    log = lambda *a: print(f"[trt-job {time.perf_counter() - t0:6.0f}s | RSS {_rss_gb():.1f} GB]", *a, flush=True)
    from ultralytics.data.utils import check_det_dataset
    dev = torch.device("cpu")
    DATA = check_det_dataset(job["data_yaml"])
    # pin_memory=False: pinned memory membuat context CUDA PyTorch -> dua context (torch + pycuda) di satu
    # proses -> "CUDA error: invalid resource handle" (penyebab kernel mati di Kaggle, 10 Okt).
    loader = make_loader(DATA, job["split"], 1, job["imgsz"], workers=2, pin_memory=False)
    EV = {k: job[k] for k in ("data_yaml", "conf", "iou", "max_det", "imgsz")}; EV["report_split"] = job["split"]
    NS = init_trt()
    assert not torch.cuda.is_initialized(), "Context CUDA PyTorch terbentuk di proses job TRT — harus tidak ada"
    if NS is None:
        Path(job["out_json"]).write_text(json.dumps({"trt_available": False}))
        sys.exit(3)
    log("TensorRT", NS.trt.__version__)
    eng = {}
    for k, p in job["onnx"].items():
        eng[k] = str(Path(p).with_suffix(".engine"))
        NS.build_engine(p, eng[k]); log("engine", k, "OK")
    SYS = TRTSystem(NS, eng)
    from pycocotools.coco import COCO
    import contextlib, io
    with contextlib.redirect_stdout(io.StringIO()):
        gt = COCO(job["gt_json"])
    res = {"trt_available": True, "trt_version": NS.trt.__version__, "acc": {}}
    for name, fn in (("Router (TRT)", predict_router_trt(SYS)), ("Vanilla (TRT)", predict_static_trt(SYS.vanilla)),
                     ("P2-Static (TRT)", predict_static_trt(SYS.p2static))):
        r = run_eval(name, fn, loader, DATA, EV, dev, save_json=True, verbose=False)
        res["acc"][name] = {**summarize_acc(r), **{f"coco_{m}": v for m, v in coco_eval(gt, r["jdict"])[0].items()}}
        if name.startswith("Router"):
            res["files"] = [x["file"] for x in r["records"]]
            res["router_branch"] = [x["branch"] for x in r["records"]]
            res["router_gate"] = [x["gate"] for x in r["records"]]
        log(name, "mAP50", round(res["acc"][name]["mAP50"], 4)); del r
    lat, act = benchmark_latency(SYS, loader, job["n_warmup"], job["n_reps"])
    res.update(latency=lat, activation_bench=act)
    Path(job["out_json"]).write_text(json.dumps(res, default=float))
    assert not torch.cuda.is_initialized(), "Context CUDA PyTorch terbentuk di proses job TRT"
    log("selesai")


if __name__ == "__main__":
    import sys
    trt_job_main(sys.argv[1])

In [ ]:
%%writefile /kaggle/working/thesis-Difficulty-Aware-Router/tahap2b_plafon.py
# [8b] modul Tahap 2b (hapus sel ini setelah file di-commit ke repo)
"""
tahap2b_plafon.py — Uji plafon routing + gate post-hoc (Opsi A) untuk Router-P2 Dual-Branch.

Pertanyaan:
  (1) PLAFON: adakah routing per gambar (A vs B) yang mengalahkan routing acak secara berarti pada aktivasi
      yang sama, bila target diukur dari HASIL DETEKSI (bukan loss)? Oracle memakai GT test -> batas atas,
      termasuk optimisme seleksi; bukan klaim deployment.
  (2) DAPAT DIPELAJARI: gate baru di atas detektor BEKU (fitur pooled dari keluaran Stage 1 — sudah tersedia
      di engine Stage 1, jadi deployable), dilatih di TRAIN, dipilih di VAL dengan aturan tetap, dinilai
      SEKALI di TEST.

Desain:
  - Satu forward per gambar (letterbox persegi 640, format deployment): z gate lama, fitur pooled
    (GAP+GMP dari p3_neck, p2_backbone, p4_neck, p5_sppf), deteksi pasca-NMS Branch A & B (conf 0.001,
    IoU 0.7, max_det 300 — sama dengan Tahap 2), dikonversi ke koordinat asli PER SUMBU.
  - COCOeval dijalankan SEKALI untuk A dan SEKALI untuk B per split. Karena evaluateImg COCO independen per
    gambar, metrik routing apa pun = gabungan evalImgs A/B per gambar (CocoMix) -> ratusan routing murah.
    Diverifikasi sama persis dengan COCOeval pada jdict ter-route (verify_mix).
  - Target per gambar dari evalImgs (IoU 0.5): soft-recall objek kecil (jumlah skor deteksi TP pada GT kecil),
    soft-recall semua objek, dan TP−FP semua objek pada conf ≥ 0.25. Δ = nilai A − nilai B.
"""
import copy
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

import tahap2_eval as E
from router_train_utils import find_router, _raw_gate_logit

TARGETS = ("soft_small", "soft_all", "net25_all")


# =============================================================================
# CACHE: deteksi pasca-NMS A/B (koordinat asli), z, fitur pooled
# =============================================================================
def _scale_xyxy(b, ratio_pad, ori_shape):
    (gh, gw), (px, py) = ratio_pad
    b = b.clone()
    b[:, [0, 2]] = ((b[:, [0, 2]] - px) / gw).clamp(0, ori_shape[1])
    b[:, [1, 3]] = ((b[:, [1, 3]] - py) / gh).clamp(0, ori_shape[0])
    return b


def _to_dets(y, ratio_pad, ori_shape, conf, iou, max_det):
    """y (1,4+nc,N) -> array (n,6): x, y, w, h (COCO, piksel asli), skor, category_id (=cls+1)."""
    from ultralytics.utils.nms import non_max_suppression
    x = non_max_suppression(y.clone(), conf, iou, nc=0, multi_label=True, agnostic=False, max_det=max_det)[0]
    if not len(x):
        return np.zeros((0, 6), np.float32)
    b = _scale_xyxy(x[:, :4], ratio_pad, ori_shape)
    out = torch.stack([b[:, 0], b[:, 1], b[:, 2] - b[:, 0], b[:, 3] - b[:, 1], x[:, 4], x[:, 5] + 1], 1)
    return out.float().cpu().numpy()


def _pool(t):
    return torch.cat([t.mean((2, 3)), t.amax((2, 3))], 1)


@torch.no_grad()
def cache_dets(model, stages, loader, device, conf=0.001, iou=0.7, max_det=300, max_images=None, log_every=1000):
    """Loader batch 1, urutan tetap. Returns dict file, image_id, z, feat (N,D), detA, detB (list array (n,6))."""
    s1, s2a, s2b = stages
    rt = find_router(model)
    out = dict(file=[], image_id=[], z=[], feat=[], detA=[], detB=[])
    t0 = time.perf_counter()
    for i, b in enumerate(loader):
        if max_images and i >= max_images:
            break
        assert b["img"].shape[0] == 1
        im = b["img"].to(device).float() / 255.0
        _, p3, p2, p4, p5 = s1(im)
        rp, osh = b["ratio_pad"][0], b["ori_shape"][0]
        out["detA"].append(_to_dets(s2a(p3, p2, p4, p5), rp, osh, conf, iou, max_det))
        out["detB"].append(_to_dets(s2b(p3, p4, p5), rp, osh, conf, iou, max_det))
        out["z"].append(float(_raw_gate_logit(rt, p3, p2)[0]))
        out["feat"].append(torch.cat([_pool(p3), _pool(p2), _pool(p4), _pool(p5)], 1)[0].float().cpu().numpy())
        stem = Path(b["im_file"][0]).stem
        out["file"].append(Path(b["im_file"][0]).name)
        out["image_id"].append(int(stem) if stem.isnumeric() else stem)
        if log_every and (i + 1) % log_every == 0:
            print(f"  {i + 1} gambar | {time.perf_counter() - t0:.0f}s", flush=True)
    out["z"] = np.asarray(out["z"]); out["feat"] = np.stack(out["feat"])
    return out


def jdict(dets, image_ids, use_a=None):
    """dets: dict detA/detB; use_a None -> pakai detA semua. Returns list COCO results."""
    js = []
    for i, iid in enumerate(image_ids):
        d = dets["detA"][i] if (use_a is None or use_a[i]) else dets["detB"][i]
        for x, y, w, h, s, c in d:
            js.append({"image_id": iid, "category_id": int(c), "bbox": [float(x), float(y), float(w), float(h)], "score": float(s)})
    return js


def subset_coco_gt(dataset, names, path, n):
    """GT COCO untuk n gambar pertama dataset (sesuai cache_dets dengan max_images)."""
    ds = copy.copy(dataset); ds.labels = dataset.labels[:n]
    return E.build_coco_gt(ds, names, path)


# =============================================================================
# CAMPURAN evalImgs A/B -> metrik routing apa pun
# =============================================================================
class CocoMix(E.CocoBootstrap):
    """Metrik COCO untuk routing per gambar dari evalImgs A & B (indeks = urutan E.params.imgIds)."""
    def __init__(self, bA, bB, use_a_sorted):
        for k in ("T", "R", "I", "A", "K", "maxDet", "iou50"):
            setattr(self, k, getattr(bA, k))
        assert bA.I == bB.I == len(use_a_sorted)
        self.cells = {key: [ca if u else cb for ca, cb, u in zip(bA.cells[key], bB.cells[key], use_a_sorted)]
                      for key in bA.cells}


class Router:
    """Mengelola urutan: array per gambar (urutan cache/loader) <-> urutan imgIds COCO (sorted)."""
    def __init__(self, E_A, E_B, image_ids):
        self.bA, self.bB = E.CocoBootstrap(E_A), E.CocoBootstrap(E_B)
        ids_sorted = list(E_A.params.imgIds)
        assert ids_sorted == list(E_B.params.imgIds)
        pos = {iid: j for j, iid in enumerate(ids_sorted)}
        self.perm = np.array([pos[i] for i in image_ids])          # cache i -> posisi COCO
        self.n = len(image_ids)

    def sorted_mask(self, use_a):
        m = np.zeros(self.n, bool); m[self.perm] = np.asarray(use_a, bool)
        return m

    def mix(self, use_a):
        return CocoMix(self.bA, self.bB, self.sorted_mask(use_a))

    def metrics(self, use_a, idx=None):
        return self.mix(use_a).metrics(np.arange(self.n) if idx is None else idx)

    def targets(self, conf_thr=0.25):
        """Δ per gambar (A − B), urutan cache. Juga jumlah GT kecil & semua (non-ignored)."""
        def per_image(b):
            res = {t: np.zeros(self.n) for t in TARGETS}
            res["n_gt_small"] = np.zeros(self.n); res["n_gt_all"] = np.zeros(self.n)
            t50 = b.iou50
            for (k, area), cells in b.cells.items():
                if area not in ("all", "small"):
                    continue
                for j, c in enumerate(cells):
                    if c is None:
                        continue
                    sc, dtm, dtig, npig = c
                    tp = dtm[t50] & ~dtig[t50]
                    if area == "small":
                        res["soft_small"][j] += sc[tp].sum(); res["n_gt_small"][j] += npig
                    else:
                        res["soft_all"][j] += sc[tp].sum(); res["n_gt_all"][j] += npig
                        fp = ~dtm[t50] & ~dtig[t50]
                        res["net25_all"][j] += (tp & (sc >= conf_thr)).sum() - (fp & (sc >= conf_thr)).sum()
            return res
        rA, rB = per_image(self.bA), per_image(self.bB)
        out = {t: (rA[t] - rB[t])[self.perm] for t in TARGETS}
        out["n_gt_small"] = rA["n_gt_small"][self.perm]; out["n_gt_all"] = rA["n_gt_all"][self.perm]
        return out


def verify_mix(router, coco_gt, dets, image_ids, use_a, atol=1e-9):
    """Metrik CocoMix == COCOeval langsung pada jdict ter-route (bukti campuran evalImgs sah)."""
    st, _ = E.coco_eval(coco_gt, jdict(dets, image_ids, use_a))
    m = router.metrics(use_a)
    for k, kk in (("mAP50-95", "mAP50-95"), ("mAP50", "mAP50"), ("mAP_small", "mAP_small")):
        assert abs(m[k] - st[kk]) <= atol, f"CocoMix {k} {m[k]} != COCOeval {st[kk]}"
    return m


# =============================================================================
# ROUTING DARI SKOR, ACAK, KURVA
# =============================================================================
def top_rho(score, rho, rng=None):
    """use_a = ρ·N gambar dengan skor tertinggi (seri dipecah acak tetapi deterministik)."""
    n = len(score); k = int(round(rho * n))
    rng = rng or np.random.default_rng(0)
    order = np.lexsort((rng.random(n), -np.asarray(score, float)))
    m = np.zeros(n, bool); m[order[:k]] = True
    return m


def random_masks(n, rho, n_draw, seed):
    rng = np.random.default_rng(seed)
    k = int(round(rho * n))
    out = []
    for _ in range(n_draw):
        m = np.zeros(n, bool); m[rng.choice(n, k, replace=False)] = True; out.append(m)
    return out


def curve(router, scores, rhos, n_random=20, seed=0):
    """DataFrame: per ρ dan per skor -> metrik COCO + selisih terhadap rata-rata acak (n_random undian)."""
    rows = []
    for rho in rhos:
        rnd = [router.metrics(m) for m in random_masks(router.n, rho, n_random, seed + int(rho * 1000))]
        base = {k: float(np.mean([r[k] for r in rnd])) for k in E.BOOT_METRICS}
        sd = {k: float(np.std([r[k] for r in rnd])) for k in E.BOOT_METRICS}
        rows.append(dict(rho=rho, routing="acak (rata-rata)", **base, **{f"sd_{k}": v for k, v in sd.items()}))
        for name, sc in scores.items():
            m = router.metrics(top_rho(sc, rho))
            rows.append(dict(rho=rho, routing=name, **m, **{f"gain_{k}": m[k] - base[k] for k in E.BOOT_METRICS}))
    return pd.DataFrame(rows)


def paired_bootstrap_routing(router, use_a, rho, n_random=5, n_boot=1000, seed=0):
    """Bootstrap berpasangan per gambar: routing − rata-rata acak (n_random undian), pada resample yang sama."""
    mix_a = router.mix(use_a)
    mix_r = [router.mix(m) for m in random_masks(router.n, rho, n_random, seed + 7)]
    return E.paired_bootstrap(mix_a, mix_r, n_boot=n_boot, seed=seed)


# =============================================================================
# GATE POST-HOC (detektor beku)
# =============================================================================
def candidate_models():
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import Ridge, LogisticRegression
    from sklearn.ensemble import HistGradientBoostingRegressor
    c = {}
    for a in (1.0, 10.0, 100.0, 1000.0):
        c[f"ridge_a{a:g}"] = ("reg", lambda a=a: make_pipeline(StandardScaler(), Ridge(alpha=a)))
    for C in (0.001, 0.01, 0.1):
        c[f"logreg_C{C:g}"] = ("clf", lambda C=C: make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=2000)))
    c["hgb"] = ("reg", lambda: HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05, max_leaf_nodes=15,
                                                              min_samples_leaf=40, l2_regularization=1.0, random_state=0))
    return c


def fit_score(kind, make, Xtr, ytr, X):
    m = make()
    if kind == "clf":
        yb = (ytr > 0).astype(int)
        if yb.min() == yb.max():
            return None, None
        m.fit(Xtr, yb)
        return m, m.decision_function(X)
    m.fit(Xtr, ytr)
    return m, m.predict(X)


def features(cache, with_z=True):
    X = cache["feat"]
    return np.concatenate([X, cache["z"][:, None]], 1) if with_z else X


def select_gate(cache_tr, tgt_tr, cache_va, router_va, rhos, n_random=20, seed=0, select_metric="mAP_small"):
    """
    Latih semua kandidat (model x target) di TRAIN, nilai di VAL: rata-rata gain metrik vs acak pada rhos.
    Aturan pemilihan TETAP: argmax rata-rata gain `select_metric` di val. Returns (tabel, nama terbaik, objek).
    """
    Xtr, Xva = features(cache_tr), features(cache_va)
    rnd = {rho: np.mean([router_va.metrics(m)[select_metric]
                         for m in random_masks(router_va.n, rho, n_random, seed + int(rho * 1000))]) for rho in rhos}
    rows, fitted = [], {}
    for tname in TARGETS:
        for mname, (kind, make) in candidate_models().items():
            m, s = fit_score(kind, make, Xtr, tgt_tr[tname], Xva)
            if m is None:
                continue
            gains = [router_va.metrics(top_rho(s, rho))[select_metric] - rnd[rho] for rho in rhos]
            key = f"{tname}|{mname}"
            fitted[key] = (kind, m)
            rows.append(dict(kandidat=key, target=tname, model=mname, mean_gain_val=float(np.mean(gains)),
                             **{f"gain_rho{rho:g}": g for rho, g in zip(rhos, gains)}))
    tab = pd.DataFrame(rows).sort_values("mean_gain_val", ascending=False).reset_index(drop=True)
    best = tab.kandidat.iloc[0]
    return tab, best, fitted[best]


def gate_score(fitted, cache):
    kind, m = fitted
    X = features(cache)
    return m.decision_function(X) if kind == "clf" else m.predict(X)

In [ ]:
# [9] DATA (loader persegi batch 1, urutan tetap)
import importlib, gc, numpy as np, pandas as pd, torch
import router_train_utils as U, tahap2_eval as E, tahap2b_plafon as P
importlib.reload(U); importlib.reload(E); importlib.reload(P)
from ultralytics.data.utils import check_det_dataset
DEV = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0); np.random.seed(0); torch.backends.cudnn.benchmark = False
DATA = check_det_dataset(CFG["data_yaml"]); NAMES = DATA["names"]
L = {s: E.make_loader(DATA, s, 1, CFG["imgsz"]) for s in ("train", "val", "test")}
for s in ("val", "test"):
    exp = CFG["expected_counts"].get(s)
    assert exp is None or len(L[s].dataset) == exp, f"{s}: {len(L[s].dataset)} gambar, diharapkan {exp}"
assert not (set(L["val"].dataset.im_files) & set(L["test"].dataset.im_files))
assert not (set(L["train"].dataset.im_files) & set(L["test"].dataset.im_files)), "Gambar test ada di train!"
N = {s: len(ld.dataset) if (s != "train" or not CFG["n_train"]) else min(CFG["n_train"], len(ld.dataset)) for s, ld in L.items()}
GT = {s: P.subset_coco_gt(L[s].dataset, NAMES, OUT / f"gt_{s}_coco.json", N[s]) for s in L}
print("Jumlah gambar:", N)

In [ ]:
# [10] PER SEED: cache -> COCOeval A/B -> target -> plafon -> gate post-hoc -> uji di test
import glob, contextlib, io
SCORES_ORDER = None

def run_seed(seed):
    out = OUT / f"seed{seed}"; out.mkdir(exist_ok=True)
    if (out / "DONE").exists():
        print(f"seed {seed}: sudah selesai, dilewati"); return
    t0 = time.perf_counter()
    hits = sorted(glob.glob(str(Path(CFG["archives"][seed]) / "**" / "weights" / CFG["ckpt_name"]), recursive=True))
    assert len(hits) == 1, f"{CFG['ckpt_name']} ditemukan {len(hits)}x untuk seed {seed}: {hits}"
    model, minfo = E.load_dualbranch(hits[0], CFG["model_yaml"], DATA["nc"], NAMES, DEV)
    assert minfo["train_args_seed"] in (None, seed), f"checkpoint seed {minfo['train_args_seed']} != {seed}"
    stages = E.make_stages(model)
    E.check_stage_equivalence(model, stages, L["test"], DEV)

    C, R, TG = {}, {}, {}
    for s in ("train", "val", "test"):
        print(f"[seed {seed}] cache {s} ({N[s]} gambar)", flush=True)
        C[s] = P.cache_dets(model, stages, L[s], DEV, CFG["conf"], CFG["iou"], CFG["max_det"], max_images=N[s])
        assert len(C[s]["z"]) == N[s]
        with contextlib.redirect_stdout(io.StringIO()):
            _, EA = E.coco_eval(GT[s], P.jdict(C[s], C[s]["image_id"], np.ones(N[s], bool)))
            _, EB = E.coco_eval(GT[s], P.jdict(C[s], C[s]["image_id"], np.zeros(N[s], bool)))
        R[s] = P.Router(EA, EB, C[s]["image_id"]); TG[s] = R[s].targets(); del EA, EB; gc.collect()
        mA, mB = R[s].metrics(np.ones(N[s], bool)), R[s].metrics(np.zeros(N[s], bool))
        print(f"   A: {mA} | B: {mB} | {time.perf_counter() - t0:.0f}s", flush=True)

    # bukti campuran evalImgs == COCOeval langsung (routing gate lama @ρ uji)
    ua = P.top_rho(C["test"]["z"], CFG["rho_test"])
    P.verify_mix(R["test"], GT["test"], C["test"], C["test"]["image_id"], ua)
    print("   CocoMix == COCOeval ✅")

    # korelasi target dengan GT (deskriptif) + korelasi antar-target
    from scipy.stats import spearmanr
    desc = {f"{t}_vs_n_gt_small": spearmanr(TG["test"][t], TG["test"]["n_gt_small"])[0] for t in P.TARGETS}
    desc.update({f"frac_pos_{t}": float((TG["test"][t] > 0).mean()) for t in P.TARGETS})
    desc.update({f"frac_zero_{t}": float((TG["test"][t] == 0).mean()) for t in P.TARGETS})

    # gate post-hoc: latih di train, pilih di val (aturan tetap)
    sel_tab, best, fitted = P.select_gate(C["train"], TG["train"], C["val"], R["val"], CFG["rhos"],
                                          n_random=CFG["n_random_curve"], seed=seed, select_metric=CFG["select_metric"])
    sel_tab.to_csv(out / "seleksi_gate_val.csv", index=False)
    print(f"   kandidat terbaik di val: {best} (gain rata-rata {sel_tab.mean_gain_val.iloc[0]:+.4f})")
    s_gate = {s: P.gate_score(fitted, C[s]) for s in ("val", "test")}

    # skor routing pada test (oracle memakai GT test -> plafon, bukan deployment)
    rhos_curve = sorted(set(CFG["rhos"] + [CFG["rho_test"], CFG["r_star_tahap2"][seed]]))
    scores_test = {"gate lama z": C["test"]["z"],
                   f"gate post-hoc [{best}]": s_gate["test"],
                   "heuristik GT: jumlah objek kecil": TG["test"]["n_gt_small"],
                   **{f"oracle Δ{t} (GT test)": TG["test"][t] for t in P.TARGETS}}
    scores_val = {"gate lama z": C["val"]["z"], f"gate post-hoc [{best}]": s_gate["val"],
                  "heuristik GT: jumlah objek kecil": TG["val"]["n_gt_small"],
                  **{f"oracle Δ{t} (GT val)": TG["val"][t] for t in P.TARGETS}}
    cur_t = P.curve(R["test"], scores_test, rhos_curve, n_random=CFG["n_random_curve"], seed=seed).assign(split="test")
    cur_v = P.curve(R["val"], scores_val, rhos_curve, n_random=CFG["n_random_curve"], seed=seed).assign(split="val")
    pd.concat([cur_v, cur_t]).to_csv(out / "kurva_routing.csv", index=False)

    # uji bootstrap berpasangan di test
    tests = []
    for name, sc, rho in ((f"gate post-hoc @ρ={CFG['rho_test']}", s_gate["test"], CFG["rho_test"]),
                          (f"gate post-hoc @r*={CFG['r_star_tahap2'][seed]}", s_gate["test"], CFG["r_star_tahap2"][seed]),
                          (f"gate lama z @ρ={CFG['rho_test']}", C["test"]["z"], CFG["rho_test"]),
                          (f"heuristik GT objek kecil @ρ={CFG['rho_test']}", TG["test"]["n_gt_small"], CFG["rho_test"]),
                          (f"oracle Δsoft_small @ρ={CFG['rho_test']}", TG["test"]["soft_small"], CFG["rho_test"])):
        bs = P.paired_bootstrap_routing(R["test"], P.top_rho(sc, rho), rho, n_random=CFG["n_random_boot"],
                                        n_boot=CFG["n_boot"], seed=seed)
        tests.append(bs.assign(uji=name, rho=rho))
        print(f"   {name}: " + " | ".join(f"{r.metric} {r.diff:+.4f} [{r.ci_lo:+.4f}, {r.ci_hi:+.4f}]" for r in bs.itertuples()), flush=True)
    BS = pd.concat(tests)[["uji", "rho", "metric", "diff", "ci_lo", "ci_hi", "p_two_sided", "n_boot"]]
    BS.to_csv(out / "bootstrap_test.csv", index=False)

    pd.DataFrame({"file": C["test"]["file"], "z": C["test"]["z"], "gate_posthoc": s_gate["test"],
                  **{k: v for k, v in TG["test"].items()}}).to_csv(out / "per_image_test.csv", index=False)
    pd.DataFrame({"file": C["val"]["file"], "z": C["val"]["z"], "gate_posthoc": s_gate["val"],
                  **{k: v for k, v in TG["val"].items()}}).to_csv(out / "per_image_val.csv", index=False)
    summ = dict(seed=seed, ckpt=hits[0], best_candidate=best, minutes=round((time.perf_counter() - t0) / 60, 1),
                metrics_A_test=R["test"].metrics(np.ones(N["test"], bool)),
                metrics_B_test=R["test"].metrics(np.zeros(N["test"], bool)), target_desc=desc,
                n_features=int(P.features(C["test"]).shape[1]))
    (out / "ringkasan.json").write_text(json.dumps(summ, indent=1, default=float))
    (out / "DONE").write_text(datetime.datetime.utcnow().isoformat())
    del model, stages, C, R, TG; gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f"[seed {seed}] selesai dalam {summ['minutes']} menit\n", flush=True)

for seed in CFG["archives"]:
    run_seed(seed)

In [ ]:
# [11] AGREGASI 3 SEED + PLOT + ARSIP
import matplotlib.pyplot as plt, zipfile
seeds = [s for s in CFG["archives"] if (OUT / f"seed{s}" / "DONE").exists()]
BS_ALL = pd.concat([pd.read_csv(OUT / f"seed{s}" / "bootstrap_test.csv").assign(seed=s) for s in seeds])
BS_ALL["uji_umum"] = BS_ALL.uji.str.replace(r" @r\*=[0-9.]+", " @r*", regex=True)
agg = (BS_ALL.groupby(["uji_umum", "metric"])
       .agg(diff_mean=("diff", "mean"), diff_sd=("diff", "std"), n_ci_above_0=("ci_lo", lambda x: int((x > 0).sum())),
            n_ci_below_0=("ci_hi", lambda x: int((x < 0).sum())), n_seed=("seed", "nunique")).reset_index())
print(agg.round(4).to_string(index=False)); agg.to_csv(OUT / "agregasi_bootstrap.csv", index=False)
SEL = pd.concat([pd.read_csv(OUT / f"seed{s}" / "seleksi_gate_val.csv").head(5).assign(seed=s) for s in seeds])
print(SEL[["seed", "kandidat", "mean_gain_val"]].to_string(index=False))

fig, axes = plt.subplots(1, len(seeds), figsize=(6 * len(seeds), 4.5), squeeze=False)
for ax, s in zip(axes[0], seeds):
    cur = pd.read_csv(OUT / f"seed{s}" / "kurva_routing.csv"); cur = cur[cur.split == "test"]
    for name, g in cur[cur.routing != "acak (rata-rata)"].groupby("routing"):
        ax.plot(g.rho, g.gain_mAP_small, "o-" if "oracle" not in name else "s--", label=name, alpha=.9 if "oracle" not in name else .5)
    rnd = cur[cur.routing == "acak (rata-rata)"]
    ax.fill_between(rnd.rho, -2 * rnd.sd_mAP_small, 2 * rnd.sd_mAP_small, color="grey", alpha=.2, label="acak ±2 sd")
    ax.axhline(0, c="k", lw=.8); ax.set(title=f"seed {s} — test", xlabel="aktivasi P2 ρ", ylabel="gain mAP_small vs acak")
axes[0][0].legend(fontsize=7)
plt.tight_layout(); plt.savefig(OUT / "tahap2b_kurva.png", dpi=120); plt.show()

PROV.update(finished_utc=datetime.datetime.utcnow().isoformat(timespec="seconds"),
            wall_min=round((time.perf_counter() - T_START) / 60, 1))
(OUT / "provenance.json").write_text(json.dumps(PROV, indent=1, default=str))
arc = Path("/kaggle/working/tahap2b.zip") if "CFG_OVERRIDE" not in globals() else OUT.parent / "tahap2b.zip"
with zipfile.ZipFile(arc, "w", zipfile.ZIP_DEFLATED) as z:
    for f in OUT.rglob("*"):
        if f.is_file() and not f.name.startswith("gt_"):
            z.write(f, f.relative_to(OUT))
print(f"📦 {arc} ({arc.stat().st_size / 1e6:.1f} MB) | {PROV['wall_min']} menit")